<a href="https://colab.research.google.com/github/fadilaisy/google-collab-and-stuff/blob/main/currentfigarch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install numpy==2.0.2 pandas==2.2.2 requests==2.32.4 xlsxwriter tqdm  numba

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 14.6 MB/s eta 0:00:00


In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
import sys, gc, math, time as pytime, warnings, threading, traceback, re, json
from pathlib import Path
from dataclasses import dataclass
from datetime import date, datetime, timedelta, timezone
from typing import List, Tuple, Optional, Dict, Union
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view as swv
from scipy.optimize import minimize
import requests
from requests.adapters import HTTPAdapter
try:
    from urllib3.util.retry import Retry
except Exception:
    from requests.packages.urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed
from functools import partial
from tqdm import tqdm

warnings.filterwarnings("ignore")
USE_GPU = False
cp = None
try:
    import cupy as _cp
    if _cp.cuda.runtime.getDeviceCount() > 0:
        cp = _cp
        USE_GPU = True
except Exception:
    cp = None
    USE_GPU = False
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
#
API_TOKEN    = os.environ.get("EODHD_API_TOKEN", "69b4102fe03158.38387378")
ALPHAS       = [0.01, 0.025, 0.05]
ALPHA_SFX    = [f"a{str(a).replace('0.','0p')}" for a in ALPHAS]
WIN_1Y       = 252
WIN_10Y      = 2520
AS_OF_DATE   = date.today() - timedelta(days=1)
END_DATE     = AS_OF_DATE
START_DATE   = END_DATE - timedelta(days=15*365)
END_TS  = pd.Timestamp(END_DATE)
CUT_1Y  = END_TS - pd.DateOffset(years=1)
CUT_10Y = END_TS - pd.DateOffset(years=10)
TICKERS = [
    "usdjpy.forex","usdchf.forex", "gbpchf.forex","eurgbp.forex","gbpjpy.forex",
    "nzdaud.forex","eurjpy.forex","usdcad.forex"
]

API_TIMEOUT  = (5, 25)
API_RETRIES  = 3
BACKOFF      = 0.5
CALL_SLEEP   = 0.35
DL_MAX_WORKERS = 10
COMPUTE_MAX_WORKERS = min(10, (os.cpu_count() or 2))
FALLBACK_INTRADAY_LIMIT_DAYS = 14
PREWARM_5M_DAYS     = 5
DAILY_5M_DAYS       = 7
INTRADAY_INTERVAL   = "5m"
MAX_PROBE_5M_DAYS   = 180
EPS                  = 1e-12
XI_MLE_MAX           = 0.90
EVT_XI_USE_MAX_GAS   = 0.50
EVT_XI_USE_MAX_FLU   = 0.35
MAG_CAP_GAS          = 5.0
MAG_CAP_FLU          = 3.0
K_MIN_PCT            = 0.04
K_GRID_MIN_PCT       = 0.06
K_GRID_MAX_PCT       = 0.20
K_GRID_STEP_PCT      = 0.02
STAB_ETA             = 0.15
KR_DIM = 6
KR_TAU = 1
KR_RIDGE = 1e-4
KR_HALFLIFE = 6
KR_KERNEL = "gaussian"
DATA_DIR   = Path("data_raw"); DATA_DIR.mkdir(parents=True, exist_ok=True)
STAGE1_DIR = Path("STAGE1_OUT"); STAGE1_DIR.mkdir(parents=True, exist_ok=True)

def _sym_tag(sym: str) -> str:
    return sym.upper().replace('.','_').replace('-','_')

def daily_path(sym: str) -> Path:
    return DATA_DIR / f"{_sym_tag(sym)}_daily.parquet"

def intraday_path(sym: str) -> Path:
    return DATA_DIR / f"{_sym_tag(sym)}_5m.parquet"

def stage1_path(sym: str, tag: str) -> Path:
    return STAGE1_DIR / f"{_sym_tag(sym)}__{tag}.parquet"

def stage1_json_path(sym: str, tag: str) -> Path:
    return STAGE1_DIR / f"{_sym_tag(sym)}__{tag}.json"

def _assert_pit_alignment(df: pd.DataFrame, cols: List[str], label: str) -> None:
    for c in cols:
        if c not in df.columns:
            continue
        s = pd.to_numeric(df[c], errors="coerce")
        if s.notna().any():
            if s.first_valid_index() == df.index.min():
                raise AssertionError(f"[{label}] {c} semble non-PIT (pas de NaN initial après shift).")

class APIRateLimiter:
    def __init__(self, min_interval: float = 0.35):
        self.min_interval = float(min_interval)
        self.lock = threading.Lock()
        self.next_time = 0.0
    def wait(self):
        with self.lock:
            now = pytime.time()
            wait = self.next_time - now
            if wait > 0:
                pytime.sleep(wait)
            now2 = pytime.time()
            self.next_time = max(now2, self.next_time) + self.min_interval

RATE_LIMITER = APIRateLimiter(CALL_SLEEP)

def _retry_session() -> requests.Session:
    sess = requests.Session()
    kw = dict(total=API_RETRIES, read=API_RETRIES, connect=API_RETRIES,
              backoff_factor=BACKOFF, status_forcelist=[429,500,502,503,504])
    try:
        r = Retry(**kw, allowed_methods=frozenset(["GET"]))
    except TypeError:
        r = Retry(**kw)
    adapter = HTTPAdapter(max_retries=r)
    sess.mount("http://", adapter); sess.mount("https://", adapter)
    return sess

SESSION = _retry_session()

def _eodhd(url: str) -> dict:
    RATE_LIMITER.wait()
    resp = SESSION.get(url, timeout=API_TIMEOUT)
    if resp.status_code != 200:
        raise RuntimeError(f"API {resp.status_code}: {resp.text[:200]}")
    return resp.json()

def save_parquet_safe(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=True)
    except Exception:
        path_csv = path.with_suffix(".csv")
        df.to_csv(path_csv, index=True)
        return path_csv
    return path

def load_parquet_safe(path: Path) -> pd.DataFrame:
    if path.exists():
        try:
            return pd.read_parquet(path)
        except Exception:
            csv = path.with_suffix(".csv")
            if csv.exists():
                return pd.read_csv(csv, index_col=0, parse_dates=True)
    if "_5m" in path.name:
        return pd.DataFrame()
    raise FileNotFoundError(str(path))

def trim_to_asof(df: pd.DataFrame, as_of_day: date) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    if isinstance(df.index, pd.DatetimeIndex):
        return df.loc[df.index.date <= as_of_day]
    return df

def get_daily_data(symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    url = (
        f"https://eodhd.com/api/eod/{symbol.upper()}?api_token={API_TOKEN}"
        f"&from={start_date}&to={end_date}&fmt=json"
    )
    j = _eodhd(url)
    if not isinstance(j, list) or not j:
        raise RuntimeError(f"[{symbol}] Empty daily API response.")
    df = pd.DataFrame(j)
    for c in ("open","high","low","close","adjusted_close","volume"):
        if c not in df: df[c] = np.nan
    df["date"] = pd.to_datetime(df.get("date", df.get("datetime")), utc=False)
    df = df.sort_values("date").set_index("date")
    return df

def _coerce_to_date(d: Union[str, date, datetime]) -> date:
    if isinstance(d, date) and not isinstance(d, datetime):
        return d
    if isinstance(d, datetime):
        return d.date()
    return pd.Timestamp(d).date()

def get_intraday_5m(symbol: str,
                    start_day: Union[str, date, datetime],
                    end_day:   Union[str, date, datetime]) -> pd.DataFrame:
    sday = _coerce_to_date(start_day); eday = _coerce_to_date(end_day)
    start_dt_utc = datetime.combine(sday, datetime.min.time()).replace(tzinfo=timezone.utc)
    end_dt_utc   = datetime.combine(eday + timedelta(days=1), datetime.min.time()).replace(tzinfo=timezone.utc) - timedelta(seconds=1)

    url = (
        f"https://eodhd.com/api/intraday/{symbol.upper()}?api_token={API_TOKEN}"
        f"&from={int(start_dt_utc.timestamp())}&to={int(end_dt_utc.timestamp())}"
        f"&interval={INTRADAY_INTERVAL}&fmt=json"
    )
    j = _eodhd(url)
    if not isinstance(j, list) or not j:
        return pd.DataFrame(columns=["open","high","low","close","volume"]).astype({"open":float,"high":float,"low":float,"close":float,"volume":float})
    df = pd.DataFrame(j)
    ts_field = None
    for c in ("datetime","timestamp","t","date"):
        if c in df.columns:
            ts_field = c; break
    if ts_field is None:
        raise RuntimeError(f"[{symbol}] Intraday payload without datetime-like field.")
    if np.issubdtype(df[ts_field].dtype, np.number):
        dt = pd.to_datetime(df[ts_field].astype("int64"), unit="s", utc=True).dt.tz_convert(None)
    else:
        dt = pd.to_datetime(df[ts_field], utc=True, errors="coerce").dt.tz_convert(None)
    df["datetime"] = dt
    for c in ("open","high","low","close","volume"):
        if c not in df: df[c] = np.nan
    df = df.dropna(subset=["datetime"])
    return df.sort_values("datetime").set_index("datetime")

def download_max_5m_history(symbol: str,
                            end_day: date,
                            max_probe_days: int = MAX_PROBE_5M_DAYS,
                            chunk_days: int = FALLBACK_INTRADAY_LIMIT_DAYS) -> pd.DataFrame:
    blocks = []
    miss = 0
    probed = 0
    end_ptr = end_day
    while probed < max_probe_days and miss < 2:
        span = min(chunk_days, max_probe_days - probed)
        start_ptr = end_ptr - timedelta(days=span-1)
        try:
            df = get_intraday_5m(symbol, start_ptr, end_ptr)
        except Exception as e:
            df = pd.DataFrame()
        if df.empty:
            miss += 1
        else:
            blocks.append(df)
            miss = 0
        end_ptr = start_ptr - timedelta(days=1)
        probed += span
    if not blocks:
        return pd.DataFrame()
    out = pd.concat(blocks, axis=0).sort_index()
    out = out[~out.index.duplicated(keep="last")]
    return out

def split_5m_make_daily_from_mean(ohlc_5m: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if ohlc_5m is None or ohlc_5m.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    ohlc_5m = ohlc_5m.sort_index()
    g = ohlc_5m.groupby(ohlc_5m.index.floor("D"))
    rows = []
    A_list, B_list = [], []
    for d, df in g:
        n = len(df)
        if n < 4:
            continue
        k = n // 2
        A = df.iloc[:k].copy()
        B = df.iloc[k:].copy()
        if A.empty or B.empty:
            continue
        meanA_close = float(pd.to_numeric(A["close"], errors="coerce").mean())
        lastB_close = float(pd.to_numeric(B["close"], errors="coerce").iloc[-1])
        meanB_close = float(pd.to_numeric(B["close"], errors="coerce").mean())
        retB_vsA = np.log(pd.to_numeric(B["close"], errors="coerce") / max(meanA_close, EPS))
        rows.append((
            pd.to_datetime(d),
            meanA_close,
            float(np.log(max(lastB_close, EPS) / max(meanA_close, EPS))),
            float(np.log(max(meanB_close, EPS) / max(meanA_close, EPS))),
            int(len(A)), int(len(B)),
            float(retB_vsA.mean()), float(retB_vsA.std(ddof=1)),
            float(retB_vsA.quantile(0.01)), float(retB_vsA.quantile(0.05))
        ))
        A_list.append(A)
        B_list.append(B)
    ohlcA = pd.concat(A_list, axis=0) if A_list else pd.DataFrame()
    ohlcB = pd.concat(B_list, axis=0) if B_list else pd.DataFrame()
    cols = ["MeanA_close","B_last_vs_MeanA","B_mean_vs_MeanA",
            "A_count","B_count","B_mean_ret_vs_MeanA","B_std_ret_vs_MeanA",
            "B_q01_ret_vs_MeanA","B_q05_ret_vs_MeanA"]
    daily = pd.DataFrame(rows, columns=["date"]+cols).set_index("date").sort_index()
    daily[[c + "_pit" for c in cols]] = daily[cols].shift(1)
    return ohlcA, ohlcB, daily

def _mad_vec(a):
    a = pd.to_numeric(pd.Series(a), errors="coerce").to_numpy(dtype=float)
    med = np.nanmedian(a)
    return 1.4826*np.nanmedian(np.abs(a - med))

def _robust_scale(s: pd.Series):
    s = pd.to_numeric(s, errors="coerce")
    med = s.median()
    sc  = 1.4826*np.median(np.abs(s - med))
    if not (np.isfinite(sc) and sc>0):
        sc = s.std(ddof=1)
    return float(sc if (np.isfinite(sc) and sc>0) else 1.0)

def log_returns(ohlc: pd.DataFrame, price_col: str = "close") -> pd.Series:
    cols = {c.lower(): c for c in ohlc.columns}
    col = price_col if price_col in ohlc.columns else cols.get("close", None)
    if col is None:
        num_cols = [c for c in ohlc.columns if pd.api.types.is_numeric_dtype(ohlc[c])]
        if not num_cols:
            return pd.Series(dtype=float, index=ohlc.index)
        col = num_cols[0]
    px = pd.to_numeric(ohlc[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    px = px.replace(0.0, np.nan)
    r = np.log(px / px.shift(1))
    return r.replace([np.inf, -np.inf], np.nan).dropna()

def gaussian_roll(r: pd.Series, win: int) -> pd.DataFrame:
    x = pd.to_numeric(r, errors="coerce").astype(float)
    med = x.rolling(win, min_periods=win).median()
    s_mad = x.rolling(win, min_periods=win).apply(lambda y: _mad_vec(y), raw=False).replace(0, np.nan)
    z = (x - med) / s_mad
    z = z.where(np.isfinite(z), 0.0)
    xw = x.where(z.abs() <= 8.0, med + z.clip(-8,8)*s_mad)
    m  = xw.rolling(win, min_periods=win).mean()
    sd = xw.rolling(win, min_periods=win).std(ddof=1)
    sk = xw.rolling(win, min_periods=win).skew()
    kt = xw.rolling(win, min_periods=win).kurt()
    out = pd.DataFrame({"mu": m, "sigma": sd, "skew": sk, "kurt_excess": kt})
    return out.iloc[win-1:]

def win_view(arr: np.ndarray, win: int) -> np.ndarray:
    try:
        return swv(arr, window_shape=win)
    except TypeError:
        n = arr.shape[0]
        if n < win: return np.empty((0, win), dtype=arr.dtype)
        shape = (n - win + 1, win)
        strides = (arr.strides[0], arr.strides[0])
        return np.lib.stride_tricks.as_strided(arr, shape=shape, strides=strides)

def _sigma_garman_klass_row(o, h, l, c) -> float:
    try:
        o = float(o); h = float(h); l = float(l); c = float(c)
        if min(o,h,l,c) <= 0 or not all(np.isfinite([o,h,l,c])): return np.nan
        co = math.log(c/o)
        hl = math.log(h/l)
        var = 0.5*(hl**2) - (2*math.log(2)-1)*(co**2)
        if var < 0: var = max(var, 0.0)
        return math.sqrt(var)
    except Exception:
        return np.nan

def sigma_diffusion_from_ohlc(ohlc: pd.DataFrame, win: int, halflife: Optional[int] = None) -> pd.Series:
    cols = {c.lower(): c for c in ohlc.columns}
    O = cols.get("open", None); H = cols.get("high", None); L = cols.get("low", None); C = cols.get("close", None)
    if any(v is None for v in [O, H, L, C]):
        return pd.Series(index=ohlc.index, dtype=float)
    s = ohlc[[O, H, L, C]].apply(lambda r: _sigma_garman_klass_row(r[O], r[H], r[L], r[C]), axis=1)
    s = pd.to_numeric(s, errors="coerce")
    if halflife is None:
        halflife = max(10, win // 8)
    s_sm = s.ewm(halflife=halflife, min_periods=max(10, halflife // 2)).mean()
    s_sm = s_sm.shift(1)
    s_sm.name = "sigma_diff"
    return s_sm

def compute_win_5m_pit(ohlc_5m: pd.DataFrame, prewarm_days: int, min_win: int = 64) -> Tuple[int, int]:
    if ohlc_5m is None or ohlc_5m.empty:
        return 0, 0
    counts_per_day = ohlc_5m.index.floor('D').value_counts().sort_index()
    bpd_exp = counts_per_day.expanding(min_periods=1).median().shift(1)
    if bpd_exp.dropna().empty:
        return 0, 0
    BPD = int(bpd_exp.dropna().iloc[-1])
    WIN_5M = max(min_win, BPD * prewarm_days) if BPD > 0 else 0
    return BPD, WIN_5M

def edf_var_es_from_sorted(S: np.ndarray, cS: np.ndarray, alpha: float) -> Tuple[np.ndarray,np.ndarray,np.ndarray,np.ndarray]:
    W = S.shape[1]
    jL = max(0, min(W-1, int(np.ceil(alpha*W)-1)))
    jR = max(0, min(W-1, int(np.ceil((1.0-alpha)*W)-1)))
    varL = S[:, jL]
    varR = S[:, jR]
    esL  = cS[:, jL] / (jL+1)
    esR  = (cS[:, -1] - (cS[:, jR-1] if jR>0 else 0.0)) / (W - jR)
    return varL, esL, varR, esR

def _min_exceedances(w: int) -> int:
    return max(15, int(K_MIN_PCT * w))

def _k_grid(w: int) -> List[int]:
    kmin = _min_exceedances(w)
    kmax = int(min(math.floor(0.25*w), w-5))
    ks = []
    p = K_GRID_MIN_PCT
    while p <= K_GRID_MAX_PCT + 1e-9:
        k = int(max(kmin, math.ceil(p*w)))
        if k <= kmax: ks.append(k)
        p += K_GRID_STEP_PCT
    ks = sorted(set(ks))
    if not ks: ks = [kmin]
    return ks

def _gpd_nll(params: np.ndarray, Y: np.ndarray) -> float:
    xi, beta = float(params[0]), float(params[1])
    if beta <= EPS: return np.inf
    if abs(xi) > 1e-8:
        z = (xi / beta) * Y
        t = 1.0 + z
        if np.any(t <= 0): return np.inf
        L = np.sum(np.log1p(z))
        return float(Y.size*np.log(beta) + (1.0 + 1.0/xi)*L)
    else:
        return float(Y.size*np.log(beta) + np.sum(Y)/beta)

def _gpd_fit_mle_scipy(Y: np.ndarray) -> Tuple[float, float, float]:
    ybar = float(np.mean(Y))
    var = float(np.var(Y, ddof=1)) if Y.size > 1 else np.nan
    xi0 = 0.1
    if np.isfinite(var) and var > 0:
        xi0 = 0.5 * (1.0 - (ybar*ybar / max(var, EPS)))
    xi0 = float(np.clip(xi0, -XI_MLE_MAX + 1e-6, XI_MLE_MAX - 1e-6))
    beta0 = float(max(EPS, ybar * (1.0 - xi0)))
    bounds = ((-XI_MLE_MAX, XI_MLE_MAX), (EPS, None))
    res = minimize(_gpd_nll, x0=np.array([xi0, beta0]), args=(Y,),
                   method='L-BFGS-B', bounds=bounds,
                   options={'ftol': 1e-9, 'maxiter': 200})
    if res.success:
        xi, beta = float(res.x[0]), float(res.x[1])
        final_nll = float(res.fun)
    else:
        xi, beta = xi0, beta0
        final_nll = _gpd_nll(np.array([xi, beta]), Y)
    lbar = -final_nll / max(1, len(Y))
    return float(xi), float(beta), float(lbar)

def _gpd_pwm(Y: np.ndarray) -> Tuple[float, float]:
    Ys = np.sort(np.asarray(Y, dtype=np.float64))
    m = Ys.size
    if m < 10 or not np.isfinite(Ys).all():
        return (np.nan, np.nan)
    j = np.arange(1, m+1, dtype=np.float64)
    p = (j - 0.35)/m
    a0 = float(Ys.mean()); a1 = float(np.mean(Ys * (1.0 - p)))
    den = a0 - 2.0*a1
    if not np.isfinite(den) or den <= EPS:
        return (np.nan, np.nan)
    xi   = a0/den - 2.0
    beta = 2.0*a0*a1/den
    xi = float(np.clip(xi, -XI_MLE_MAX+1e-6, XI_MLE_MAX-1e-6))
    beta = float(max(EPS, beta))
    return xi, beta

def _ks_gpd_distance(Y: np.ndarray, xi: float, beta: float) -> float:
    if Y.size < 10 or not (np.isfinite(xi) and np.isfinite(beta) and beta>0):
        return np.inf
    Ys = np.sort(Y.astype(float, copy=False))
    m  = Ys.size
    if abs(xi) < 1e-12:
        F = 1.0 - np.exp(-Ys/beta)
    else:
        t = 1.0 + (xi/beta)*Ys
        if np.any(t<=0): return np.inf
        F = 1.0 - np.power(t, -1.0/xi)
    U = np.arange(1, m+1, dtype=float) / m
    return float(np.max(np.abs(F - U)))

def _k_grid_two_stage(w: int, side: str) -> Tuple[List[int], List[int]]:
    ks_fine = _k_grid(w)
    step = max(0.04, 2.0*K_GRID_STEP_PCT)
    kmin = _min_exceedances(w)
    if side == "left":
        kmin = max(kmin, max(30, int(math.ceil(0.08*w))))
    ks_coarse = []
    p = K_GRID_MIN_PCT
    while p <= K_GRID_MAX_PCT + 1e-9:
        k = int(max(kmin, math.ceil(p*w)))
        if k < w-5:
            ks_coarse.append(k)
        p += step
    ks_coarse = sorted(set(ks_coarse))
    return ks_coarse, ks_fine

def _fit_best_gpd_on_sorted(S_sorted: np.ndarray, side: str) -> Tuple[float,float,float,float]:
    W = int(S_sorted.size)
    if W < 64:
        return (np.nan, np.nan, np.nan, np.nan)
    ks_coarse, _ = _k_grid_two_stage(W, side)
    if not ks_coarse:
        return (np.nan, np.nan, np.nan, np.nan)

    cand = []
    for K in ks_coarse:
        i_thr = W - K - 1
        if i_thr < 0 or i_thr >= W-1:
            continue
        u  = float(S_sorted[i_thr])
        Y  = S_sorted[(i_thr+1):] - u
        if Y.size < _min_exceedances(W) or not np.isfinite(Y).all():
            continue
        xi0, b0 = _gpd_pwm(Y)
        if not (np.isfinite(xi0) and np.isfinite(b0) and b0>0):
            continue
        nll0 = _gpd_nll(np.array([xi0,b0]), Y)
        lbar0 = float(-nll0/max(1,len(Y))) if np.isfinite(nll0) else -np.inf
        ksD   = _ks_gpd_distance(Y, xi0, b0)
        cand.append((K,u,xi0,b0,lbar0,ksD))

    if not cand:
        K = _min_exceedances(W)
        i_thr = W - K - 1
        if i_thr < 0 or i_thr >= W-1:
            return (np.nan, np.nan, np.nan, np.nan)
        u  = float(S_sorted[i_thr])
        Y  = S_sorted[(i_thr+1):] - u
        xi0, b0 = _gpd_pwm(Y)
        xi  = 0.1 if not np.isfinite(xi0) else float(xi0)
        beta= float(max(EPS, (np.mean(Y) if Y.size else 1.0)*(1.0 - xi)))
        return (u, K/float(W), xi, beta)

    xi_arr = np.array([c[2] for c in cand], dtype=float)
    stab = np.zeros_like(xi_arr)
    for j in range(len(xi_arr)):
        d1 = (xi_arr[j] - xi_arr[j-1])**2 if j-1>=0 else 1.0
        d2 = (xi_arr[j] - xi_arr[j+1])**2 if j+1<len(xi_arr) else 1.0
        stab[j] = d1 + d2
    ks_w = 0.6 if side=="left" else 0.5
    lbar_arr = np.array([c[4] for c in cand], dtype=float)
    ks_arr   = np.array([c[5] for c in cand], dtype=float)
    Jc = stab - STAB_ETA*lbar_arr + ks_w*ks_arr
    order = np.argsort(Jc)[:2] if Jc.size>0 else np.array([], dtype=int)

    best_J = np.inf
    best_payload = None
    for j in order:
        K,u,xi0,b0,_,_ = cand[j]
        i_thr = W - K - 1
        if i_thr < 0 or i_thr >= W-1:
            continue
        Y = S_sorted[(i_thr+1):] - u
        if Y.size < 5:
            continue
        xi, beta, lbar = _gpd_fit_mle_scipy(Y)
        ksD = _ks_gpd_distance(Y, xi, beta)
        if not np.isfinite(ksD): ksD = 1e9
        if not np.isfinite(lbar): lbar = -1e9
        J   = 0.15*(xi - xi0)**2 - STAB_ETA*lbar + ks_w*ksD
        if not np.isfinite(J): J = 1e9
        if J < best_J:
            best_J = J
            best_payload = (u, K/float(W), float(xi), float(beta))

    if best_payload is None:
        j0 = int(np.argmin(Jc)) if Jc.size>0 else 0
        K,u,xi0,b0,_,_ = cand[j0]
        return (float(u), float(K/float(W)), float(xi0), float(b0))
    return best_payload

def evt_params_from_series(r: pd.Series, win: int) -> pd.DataFrame:
    arr = r.to_numpy(dtype=np.float64)
    out = pd.DataFrame(columns=[
        "EVT_left_u","EVT_left_xi","EVT_left_beta","EVT_left_pu",
        "EVT_right_u","EVT_right_xi","EVT_right_beta","EVT_right_pu"
    ], index=r.index)
    if arr.size < win: return out
    X = win_view(arr, win)
    N, W = X.shape
    uL = np.empty(N); xiL = np.empty(N); beL = np.empty(N); puL = np.empty(N)
    uR = np.empty(N); xiR = np.empty(N); beR = np.empty(N); puR = np.empty(N)
    for i in range(N):
        row = X[i, :].astype(np.float64, copy=False)
        S_right = np.sort(row)
        ur, pur, xir, ber = _fit_best_gpd_on_sorted(S_right, side="right")
        uR[i], puR[i], xiR[i], beR[i] = ur, pur, xir, ber
        S_left  = -S_right[::-1]
        ul, pul, xil, bel = _fit_best_gpd_on_sorted(S_left, side="left")
        uL[i], puL[i], xiL[i], beL[i] = ul, pul, xil, bel
    idx = r.index[win-1:]
    return pd.DataFrame({
        "EVT_left_u":uL, "EVT_left_xi":xiL, "EVT_left_beta":beL, "EVT_left_pu":puL,
        "EVT_right_u":uR,"EVT_right_xi":xiR,"EVT_right_beta":beR,"EVT_right_pu":puR
    }, index=idx)

def evt_quant_es(u: float, xi: float, beta: float, pu: float, alpha: float, side: str) -> Tuple[float,float]:
    if not (np.isfinite(u) and np.isfinite(xi) and np.isfinite(beta) and np.isfinite(pu)):
        return (np.nan, np.nan)
    if pu <= 0 or alpha >= pu or beta <= 0:
        return (np.nan, np.nan)
    if abs(xi) < 1e-12:
        var_evt = u + beta * math.log(max(pu/alpha, 1.0+1e-12))
        es_evt  = var_evt + beta
    else:
        if xi >= 1.0: return (np.nan, np.nan)
        term = alpha/pu
        if term <= 0: return (np.nan, np.nan)
        var_evt = u + (beta/xi) * (term**(-xi) - 1.0)
        es_evt  = var_evt + (beta + xi*(var_evt - u)) / (1.0 - xi)
        if xi < 0:
            x_max = u - beta/xi
            if var_evt >= x_max:
                return (np.nan, np.nan)
    if side == "left": return (-var_evt, -es_evt)
    return (var_evt, es_evt)

def _expand_qnorm(series: pd.Series, ql=0.10, qh=0.90, minp: int=WIN_1Y) -> pd.Series:
    lo = series.expanding(min_periods=minp).quantile(ql)
    hi = series.expanding(min_periods=minp).quantile(qh)
    denom = (hi - lo).replace(0, np.nan)
    z = (series - lo) / denom
    return z.replace([np.inf,-np.inf], np.nan).clip(0.0, 1.0).fillna(0.5)

def _dfa1_window(y: np.ndarray) -> float:
    y = np.asarray(y, dtype=float)
    n = y.size
    if n < 128 or not np.isfinite(y).all():
        return np.nan
    y = y - np.median(y)
    X = np.cumsum(y)
    scales = [8,12,16,24,32,48,64,96,128]
    Fs, Ss = [], []
    for s in scales:
        m = len(X)//s
        if m < 4:
            continue
        seg = X[:m*s].reshape(m, s)
        t = np.arange(s, dtype=float)
        A = np.vstack([t, np.ones_like(t)]).T
        coef, *_ = np.linalg.lstsq(A, seg.T, rcond=None)
        res = seg.T - (A @ coef)
        F2 = np.mean(res**2, axis=0)
        F  = np.sqrt(np.maximum(F2, EPS))
        Fs.append(np.median(F))
        Ss.append(s)
    if len(Fs) < 4:
        return np.nan
    Fs = np.asarray(Fs); Ss = np.asarray(Ss, dtype=float)
    mask = np.isfinite(Fs) & (Fs > 0)
    if mask.sum() < 4:
        return np.nan
    H, _ = np.polyfit(np.log(Ss[mask]), np.log(Fs[mask]), 1)
    return float(np.clip(H if np.isfinite(H) else np.nan, 0.0, 1.0))

def rolling_hurst(r: pd.Series, win: int) -> pd.Series:
    arr = r.to_numpy(dtype=np.float64)
    out = pd.Series(np.nan, index=r.index, name=f"Hurst_W{win}")
    if arr.size < win: return out
    X = win_view(arr, win)
    vals = np.array([_dfa1_window(X[i, :]) for i in range(X.shape[0])], dtype=float)
    out.iloc[win-1:] = vals
    return out

def _mahalanobis2_rows(Sw: np.ndarray, x0w: np.ndarray, VI: np.ndarray) -> np.ndarray:
    d = Sw - x0w
    return np.einsum('ij,jk,ik->i', d, VI, d)

def _local_linear_kernel_grad(S: np.ndarray, y: np.ndarray, x0: np.ndarray,
                              h: float, lam: float, kernel: str="gaussian") -> Tuple[np.ndarray, float]:
    med = np.nanmedian(S, axis=0)
    sc  = 1.4826*np.nanmedian(np.abs(S - med), axis=0)
    sc[~np.isfinite(sc) | (sc <= EPS)] = 1.0
    Sw  = (S - med)/sc
    x0w = (x0 - med)/sc
    C = np.cov(Sw.T) if Sw.shape[0] > Sw.shape[1] else (Sw.T @ Sw)/max(1, Sw.shape[0]-1)
    C = C + (1e-6 + lam)*np.eye(Sw.shape[1])
    try:
        VI = np.linalg.pinv(C)
    except np.linalg.LinAlgError:
        VI = np.eye(Sw.shape[1])

    d2 = _mahalanobis2_rows(Sw, x0w, VI)
    if kernel == "epanechnikov":
        w = np.maximum(0.0, 1.0 - d2 / max(h*h, EPS))
    else:
        w = np.exp(-0.5 * d2 / max(h*h, EPS))
    w[w < 1e-12] = 0.0
    if np.sum(w) < 1e-8:
        return np.zeros(S.shape[1], dtype=float), float(np.median(y))

    X = np.hstack([np.ones((S.shape[0],1)), Sw])
    W = np.diag(w)
    XtW = X.T @ W
    A = XtW @ X + lam*np.eye(X.shape[1])
    b = XtW @ y
    try:
        beta = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        beta = np.linalg.lstsq(A, b, rcond=None)[0]
    grad_sw = beta[1:]
    grad = grad_sw / sc
    return grad.astype(float), float(beta[0])

def _kernel_bandwidth_auto(S: np.ndarray, x0: np.ndarray) -> float:
    med = np.nanmedian(S, axis=0)
    sc  = 1.4826*np.nanmedian(np.abs(S - med), axis=0)
    sc[~np.isfinite(sc) | (sc <= EPS)] = 1.0
    Sw  = (S - med)/sc
    x0w = (x0 - med)/sc
    C = np.cov(Sw.T) if Sw.shape[0] > Sw.shape[1] else (Sw.T @ Sw)/max(1, Sw.shape[0]-1)
    C = C + 1e-6*np.eye(Sw.shape[1])
    try:
        VI = np.linalg.pinv(C)
    except np.linalg.LinAlgError:
        VI = np.eye(Sw.shape[1])
    d2 = _mahalanobis2_rows(Sw, x0w, VI)
    d2p = d2[np.isfinite(d2) & (d2 > 0)]
    if d2p.size == 0:
        return 1.0
    return float(np.quantile(d2p, 0.25))

def _jacobian_from_grad(grad: np.ndarray) -> np.ndarray:
    m = grad.size
    J = np.zeros((m, m), dtype=float)
    J[0, :] = grad
    if m > 1:
        J[1:, :-1] = np.eye(m-1)
    return J

def rolling_lyap_kernel(r: pd.Series, win: int, m: int=KR_DIM, tau: int=KR_TAU,
                        lam: float=KR_RIDGE, kernel: str=KR_KERNEL,
                        hl: int=KR_HALFLIFE) -> pd.Series:
    arr = pd.to_numeric(r, errors="coerce").to_numpy(dtype=np.float64)
    out = pd.Series(np.nan, index=r.index, name=f"Lyap_KR_W{win}")
    if arr.size < max(win, (m-1)*tau + 2):
        return out

    Xwin = win_view(arr, win)
    idx_w = r.index[win-1:]
    vals = np.full(Xwin.shape[0], np.nan, dtype=float)

    for i in range(Xwin.shape[0]):
        x = Xwin[i, :]
        T = x.size
        if T <= (m-1)*tau + 2:
            continue
        S_rows, Y_vals = [], []
        for t in range((m-1)*tau, T-2):
            S_rows.append([x[t - k*tau] for k in range(m)])
            Y_vals.append(x[t+1])
        S = np.asarray(S_rows, dtype=float)
        Y = np.asarray(Y_vals, dtype=float)
        if S.shape[0] < 2 or S.shape[1] != m:
            continue
        x0 = np.array([x[T-2 - k*tau] for k in range(m)], dtype=float)

        h2 = _kernel_bandwidth_auto(S, x0)
        h  = math.sqrt(max(h2, EPS))
        grad, _ = _local_linear_kernel_grad(S, Y, x0, h=h, lam=lam, kernel=kernel)
        J = _jacobian_from_grad(grad)
        try:
            rho = float(np.max(np.abs(np.linalg.eigvals(J))))
        except np.linalg.LinAlgError:
            rho = np.nan
        vals[i] = float(np.log(max(rho, EPS))) if np.isfinite(rho) else np.nan

    s = pd.Series(vals, index=idx_w)
    if hl and hl > 1:
        s = s.ewm(halflife=hl, min_periods=1).mean()
    out.iloc[win-1:] = s.values
    return out

def _local_linear_kernel_predict(S: np.ndarray, y: np.ndarray, x0: np.ndarray,
                                 h: float, lam: float, kernel: str="gaussian") -> float:
    med = np.nanmedian(S, axis=0)
    sc  = 1.4826*np.nanmedian(np.abs(S - med), axis=0)
    sc[~np.isfinite(sc) | (sc <= EPS)] = 1.0
    Sw  = (S - med)/sc
    x0w = (x0 - med)/sc
    C = np.cov(Sw.T) if Sw.shape[0] > Sw.shape[1] else (Sw.T @ Sw)/max(1, Sw.shape[0]-1)
    C = C + (1e-6 + lam)*np.eye(Sw.shape[1])
    try:
        VI = np.linalg.pinv(C)
    except np.linalg.LinAlgError:
        VI = np.eye(Sw.shape[1])

    d2 = _mahalanobis2_rows(Sw, x0w, VI)
    if kernel == "epanechnikov":
        w = np.maximum(0.0, 1.0 - d2 / max(h*h, EPS))
    else:
        w = np.exp(-0.5 * d2 / max(h*h, EPS))
    w[w < 1e-12] = 0.0
    if np.sum(w) < 1e-8:
        return float(np.median(y))

    X = np.hstack([np.ones((S.shape[0],1)), Sw])
    W = np.diag(w)
    XtW = X.T @ W
    A = XtW @ X + lam*np.eye(X.shape[1])
    b = XtW @ y
    try:
        beta = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        beta = np.linalg.lstsq(A, b, rcond=None)[0]
    yhat = float(beta[0] + np.dot(beta[1:], x0w))
    return yhat

def _window_last_errors_AR_vs_Kernel(x: np.ndarray, p: int=6,
                                     lam: float=KR_RIDGE, kernel: str=KR_KERNEL) -> Tuple[float, float]:
    W = x.size
    if W <= p+1:
        return (np.nan, np.nan)
    rows, ys = [], []
    for t in range(p, W):
        rows.append([x[t-k-1] for k in range(p)])
        ys.append(x[t])
    X = np.asarray(rows, dtype=float)
    Y = np.asarray(ys, dtype=float)
    if X.shape[0] < 2:
        return (np.nan, np.nan)
    Xtr, Ytr = X[:-1, :], Y[:-1]
    Xte, Yte = X[-1, :], Y[-1]
    try:
        coef, *_ = np.linalg.lstsq(Xtr, Ytr, rcond=None)
        yhat_lin = float(np.dot(Xte, coef))
    except np.linalg.LinAlgError:
        yhat_lin = float(np.mean(Ytr))
    err_AR = float((Yte - yhat_lin)**2)
    h2 = _kernel_bandwidth_auto(Xtr, Xte); h = math.sqrt(max(h2, EPS))
    yhat_nl = _local_linear_kernel_predict(Xtr, Ytr, Xte, h=h, lam=lam, kernel=kernel)
    err_K   = float((Yte - yhat_nl)**2)
    return (err_AR, err_K)

def classify_gas_fluid_linear_nonlin(r: pd.Series, win: int, p_ar: int = 6,
                                     lam: float = KR_RIDGE, kernel: str = KR_KERNEL,
                                     rel_gain_on: float = 0.05, rel_gain_off: float = 0.02) -> pd.DataFrame:
    arr = pd.to_numeric(r, errors="coerce").to_numpy(dtype=np.float64)
    out_idx = r.index
    if arr.size < win:
        return pd.DataFrame({"p_gas": pd.Series(dtype=float), "regime": pd.Series(dtype=object)})
    X = win_view(arr, win)
    N = X.shape[0]
    idx = out_idx[win-1:]
    gains = np.full(N, np.nan, dtype=float)
    for i in range(N):
        w_hist = X[i, :-1]  # PIT
        if w_hist.size <= p_ar + 1:
            continue
        err_AR, err_K = _window_last_errors_AR_vs_Kernel(w_hist, p=p_ar, lam=lam, kernel=kernel)
        if np.isfinite(err_AR) and np.isfinite(err_K) and err_AR > EPS:
            gains[i] = (err_AR - err_K) / max(err_AR, EPS)

    s = pd.Series(gains, index=idx)
    s_sm = s.ewm(halflife=5, min_periods=1).mean()
    pg = (1.0 - s_sm.clip(lower=0.0, upper=0.20) / 0.20).clip(0.0, 1.0)
    regime = pd.Series(index=idx, dtype=object)
    state = "GAS"
    for t, g in enumerate(s_sm):
        g = float(g) if np.isfinite(g) else 0.0
        if g >= rel_gain_on and state != "FLUIDE":
            state = "FLUIDE"
        elif g <= rel_gain_off and state != "GAS":
            state = "GAS"
        regime.iloc[t] = state

    return pd.DataFrame({"p_gas": pg.fillna(1.0), "regime": regime}, index=idx)

def _detect_price_col(df: pd.DataFrame) -> str:
    if "close" in df.columns:
        return "close"
    if "adjusted_close" in df.columns:
        return "adjusted_close"
    return df.columns[0]

def _ohlc_diag(ohlc: pd.DataFrame, win: int=21) -> pd.DataFrame:
    cols = {c.lower(): c for c in ohlc.columns}
    price = cols.get("close", cols.get("adjusted_close", None))
    open_c = cols.get("open", None)
    high_c = cols.get("high", None)
    low_c  = cols.get("low",  None)
    if price is None:
        return pd.DataFrame(index=ohlc.index, columns=["chaos_score","trend_r2","regime"])
    close = pd.to_numeric(ohlc[price], errors="coerce")
    prev_close = close.shift(1)
    gap = ((pd.to_numeric(ohlc.get(open_c, close), errors="coerce") - prev_close) / prev_close)
    rng = ((pd.to_numeric(ohlc.get(high_c, close), errors="coerce") - pd.to_numeric(ohlc.get(low_c, close), errors="coerce")) / prev_close)
    ret = np.log(close/prev_close)
    gap_s = _mad_vec(gap); rng_s = _mad_vec(rng); ret_s = _mad_vec(ret)
    jump = (gap.abs() > 3.0 * (gap_s if (np.isfinite(gap_s) and gap_s>0) else gap.std(ddof=1))).astype(float)
    longc= (rng > 3.0 * (rng_s if (np.isfinite(rng_s) and rng_s>0) else rng.std(ddof=1))).astype(float)
    bigr = (ret.abs() > 3.0 * (ret_s if (np.isfinite(ret_s) and ret_s>0) else ret.std(ddof=1))).astype(float)
    x = np.arange(len(close), dtype=float)
    r2 = np.full_like(x, np.nan, dtype=float)
    for i in range(win-1, len(close)):
        sl = slice(i-win+1, i+1)
        yv = close.iloc[sl].astype(float).values
        if np.any(~np.isfinite(yv)):
            continue
        xv = x[sl]
        A = np.vstack([xv, np.ones_like(xv)]).T
        beta, *_ = np.linalg.lstsq(A, yv, rcond=None)
        yhat = A @ beta
        ss_res = np.sum((yv - yhat)**2)
        ss_tot = np.sum((yv - yv.mean())**2)
        r2[i] = 1.0 - (ss_res/ss_tot) if ss_tot>0 else np.nan
    r2s = pd.Series(r2, index=ohlc.index)
    chaos = (0.4*jump.rolling(win,2).mean() + 0.4*longc.rolling(win,2).mean() + 0.2*bigr.rolling(win,2).mean()).clip(0,1)
    reg = pd.Series(index=ohlc.index, dtype=object)
    for i, (cs, r2v) in enumerate(zip(chaos.values, r2s.values)):
        if not np.isfinite(cs):
            reg.iloc[i] = np.nan; continue
        if cs >= 0.35 or (np.isfinite(r2v) and r2v < 0.45):
            reg.iloc[i] = "FLUIDE"
        elif np.isfinite(r2v) and r2v >= 0.55 and cs < 0.35:
            reg.iloc[i] = "GAS"
        else:
            reg.iloc[i] = reg.iloc[i-1] if i>0 and pd.notna(reg.iloc[i-1]) else "GAS"
    return pd.DataFrame({"chaos_score": chaos, "trend_r2": r2s, "regime": reg}, index=ohlc.index)

def premice_signals(M: pd.DataFrame, EVT: pd.DataFrame, H: pd.Series, L: pd.Series,
                    win_long: int, win_short: int) -> pd.DataFrame:
    M = M.copy()
    if "kurt_excess" not in M.columns and "kurt_exces" in M.columns:
        M = M.rename(columns={"kurt_exces": "kurt_excess"})
    if "kurt_excess" in M.columns:
        M["kurt_excess"] = pd.to_numeric(M["kurt_excess"], errors="coerce").shift(1)
    EVT = EVT.shift(1)
    H   = pd.to_numeric(H, errors="coerce").shift(1)
    L   = pd.to_numeric(L, errors="coerce").shift(1)

    idx = M.index.intersection(EVT.index).intersection(H.index).intersection(L.index)
    if len(idx) == 0:
        return pd.DataFrame(index=M.index, columns=["premice_to_FLUIDE_score","premice_to_GAS_score"])

    kurt  = pd.to_numeric(M.get("kurt_excess"), errors="coerce").reindex(idx).clip(lower=0.0).fillna(0.0)
    xiabs = np.maximum(
        pd.to_numeric(EVT.get("EVT_left_xi"),  errors="coerce").reindex(idx).abs(),
        pd.to_numeric(EVT.get("EVT_right_xi"), errors="coerce").reindex(idx).abs()
    ).fillna(0.0)
    Hc    = pd.to_numeric(H, errors="coerce").reindex(idx).fillna(0.5)
    Ly    = pd.to_numeric(L, errors="coerce").reindex(idx)

    def norm_diff(x: pd.Series) -> pd.Series:
        mL = x.rolling(win_long,  min_periods=max(10, win_long//5)).median()
        mS = x.rolling(win_short, min_periods=max( 5, win_short//5)).median()
        d  = (mS - mL)
        sd_d = d.rolling(win_long, min_periods=max(10, win_long//5)).std().replace(0.0, np.nan)
        return _expand_qnorm((d / sd_d).abs(), minp=max(10, win_short//2))

    s_x = norm_diff(xiabs)
    s_k = norm_diff(kurt)
    s_h = _expand_qnorm((Hc-0.5).abs(), minp=max(10, win_short//2))
    muL = Ly.rolling(win_short, min_periods=max(5, win_short//5)).mean()
    sdL = Ly.rolling(win_short, min_periods=max(5, win_short//5)).std(ddof=1).replace(0.0, np.nan)
    zL  = ((Ly-muL)/sdL).clip(-5,5).abs().fillna(0.0)
    s_l = _expand_qnorm(zL, minp=max(10, win_short//2))
    to_fluid = (0.35*s_x + 0.35*s_k + 0.20*s_h + 0.10*s_l).clip(0.0,1.0)
    to_gas   = (0.40*(1.0-s_x) + 0.40*(1.0-s_k) + 0.10*(1.0-s_h) + 0.10*(1.0-s_l)).clip(0.0,1.0)
    return pd.DataFrame({
        "premice_to_FLUIDE_score": to_fluid,
        "premice_to_GAS_score":    to_gas
    }, index=idx).reindex(M.index).fillna(0.5)

def _digamma(x: float) -> float:
    if x <= 0: return np.nan
    y = x
    r = 0.0
    while y < 8.0:
        r -= 1.0/y
        y += 1.0
    f = 1.0/y
    r += math.log(y) - 0.5*f - f*f*(1.0/12.0 - f*f*(1.0/120.0 - f*f*(1.0/252.0)))
    return r

def _entropy_knn_body(x: np.ndarray, uL: float, uR: float, k: int) -> float:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if x.size <= max(3, k+1):
        return np.nan
    x = np.sort(x)
    n = x.size
    idx = np.arange(n)
    left_k  = np.maximum(0, idx - k)
    right_k = np.minimum(n-1, idx + k)
    eps = np.minimum(x[idx] - x[left_k], x[right_k] - x[idx])
    eps = np.maximum(eps, EPS)
    left_room  = x + float(uL)
    right_room = float(uR) - x
    V = np.minimum(eps, left_room) + np.minimum(eps, right_room)
    V = np.maximum(V, EPS)
    return float(_digamma(n) - _digamma(k) + np.mean(np.log(V)))

def _entropy_gpd(xi: float, beta: float) -> float:
    if not (np.isfinite(xi) and np.isfinite(beta) and beta>0.0 and xi>-1.0):
        return np.nan
    return float(math.log(beta) + 1.0 + xi)

def entropy_mixture_series(r: pd.Series, win: int, evt_df: pd.DataFrame,
                           xi_shrink: Optional[float] = None) -> pd.DataFrame:
    idx = r.index
    out = pd.DataFrame(index=idx, columns=[
        "entropy_total","entropy_body","entropy_left","entropy_right",
        "w_left","w_body","w_right","u_left","u_right",
        "xi_left","xi_right","beta_left","beta_right"
    ])
    if len(r) < win: return out

    arr = r.to_numpy(dtype=np.float64)
    X = win_view(arr, win)
    idx_w = r.index[win-1:]
    E = evt_df.reindex(idx_w).shift(1)

    for i, t in enumerate(idx_w):
        if t not in E.index:
            continue
        try:
            uL = float(E.at[t, "EVT_left_u"]);  uR = float(E.at[t, "EVT_right_u"])
            xiL= float(E.at[t, "EVT_left_xi"]); xiR= float(E.at[t, "EVT_right_xi"])
            bL = float(E.at[t, "EVT_left_beta"]); bR= float(E.at[t, "EVT_right_beta"])
            pL = float(E.at[t, "EVT_left_pu"]);   pR= float(E.at[t, "EVT_right_pu"])
        except Exception:
            continue

        xi_gate = EVT_XI_USE_MAX_GAS if xi_shrink is None else min(EVT_XI_USE_MAX_GAS, xi_shrink)
        okL = (abs(xiL) < xi_gate and bL > 0 and 0 < pL < 1)
        okR = (abs(xiR) < xi_gate and bR > 0 and 0 < pR < 1)
        wL, wR = (pL if okL else 0.0), (pR if okR else 0.0)
        wB = max(0.0, 1.0 - wL - wR)
        xw = X[i, :-1]
        body = xw[(xw > -uL) & (xw < uR)]
        hB = _entropy_knn_body(body, uL, uR, k=max(2, min(50, int(np.sqrt(max(1, body.size)))))) if body.size >= 5 else np.nan
        hL = _entropy_gpd(xiL, bL) if okL and np.isfinite(xiL) else np.nan
        hR = _entropy_gpd(xiR, bR) if okR and np.isfinite(xiR) else np.nan

        if not np.isfinite(hL):  wB += wL; wL = 0.0
        if not np.isfinite(hR):  wB += wR; wR = 0.0

        s = wL + wB + wR
        if not (np.isfinite(s) and s > 0): wL = wR = 0.0; wB = 1.0; s = 1.0
        wL /= s; wB /= s; wR /= s

        Hw = 0.0
        for w in (wL, wB, wR):
            if w > 0: Hw -= w * math.log(w)
        htot = Hw
        if np.isfinite(hB): htot += wB * hB
        if np.isfinite(hL): htot += wL * hL
        if np.isfinite(hR): htot += wR * hR

        out.loc[t, [
            "entropy_total","entropy_body","entropy_left","entropy_right",
            "w_left","w_body","w_right","u_left","u_right",
            "xi_left","xi_right","beta_left","beta_right"
        ]] = [htot, hB, hL, hR, wL, wB, wR, uL, uR, xiL, xiR, bL, bR]
    return out

def sanity_validate_entropy(df: pd.DataFrame, label: str) -> None:
    def col(name):
        return name if name in df.columns else f"ENT_{name}"
    needed = {col("w_left"), col("w_body"), col("w_right"),
              col("entropy_total"), col("entropy_body"), col("entropy_left"), col("entropy_right")}
    if not needed.issubset(df.columns):
        return
    wL = pd.to_numeric(df[col("w_left")], errors="coerce").fillna(0.0)
    wB = pd.to_numeric(df[col("w_body")], errors="coerce").fillna(0.0)
    wR = pd.to_numeric(df[col("w_right")], errors="coerce").fillna(0.0)
    Ht = pd.to_numeric(df[col("entropy_total")], errors="coerce")
    HB = pd.to_numeric(df[col("entropy_body")], errors="coerce")
    HL = pd.to_numeric(df[col("entropy_left")], errors="coerce")
    HR = pd.to_numeric(df[col("entropy_right")], errors="coerce")
    s = (wL + wB + wR)
    if not np.allclose(s.values, np.ones_like(s.values), atol=5e-6):
        raise AssertionError(f"[{label}] ENT: poids non normalisés (min={s.min():.6f}, max={s.max():.6f})")

    Hw = np.zeros_like(s.values, dtype=float)
    for i, (wl, wb, wr) in enumerate(zip(wL.values, wB.values, wR.values)):
        x = 0.0
        for w in (wl,wb,wr):
            if w>0: x -= w*math.log(w)
        Hw[i] = x
    rhs = Hw + (wB.values*np.nan_to_num(HB.values, nan=0.0)) \
              + (wL.values*np.nan_to_num(HL.values, nan=0.0)) \
              + (wR.values*np.nan_to_num(HR.values, nan=0.0))
    valid_idx = Ht.notna().values
    if not np.allclose(Ht.values[valid_idx], rhs[valid_idx], rtol=5e-3, atol=5e-3):
        diff = np.abs(Ht.values - rhs)
        raise AssertionError(f"[{label}] ENT: total ≠ Hw + somme pondérée (écart max={np.nanmax(diff):.4e})")

def sanity_validate_monotonicity(df: pd.DataFrame, label: str) -> None:
    def sort_key(col_name):
        try:
            return float(col_name.split('_')[-1].replace('a','').replace('p','.'))
        except:
            return -1
    vL_cols = sorted([c for c in df.columns if c.startswith("VaR_left_a")], key=sort_key)
    eL_cols = sorted([c for c in df.columns if c.startswith("ES_left_a")], key=sort_key)
    vR_cols = sorted([c for c in df.columns if c.startswith("VaR_right_a")], key=sort_key)
    eR_cols = sorted([c for c in df.columns if c.startswith("ES_right_a")], key=sort_key)
    vL_list = [pd.to_numeric(df[c], errors="coerce") for c in vL_cols]
    eL_list = [pd.to_numeric(df[c], errors="coerce") for c in eL_cols]
    vR_list = [pd.to_numeric(df[c], errors="coerce") for c in vR_cols]
    eR_list = [pd.to_numeric(df[c], errors="coerce") for c in eR_cols]
    def _check(lst, sign, side):
        if not lst or len(lst) < 2: return
        arr = np.vstack([x.values for x in lst]).T
        d = np.diff(arr, axis=1)
        ok = (d*sign >= -1e-9)
        if not np.all(ok):
            raise AssertionError(f"[{label}] {side} tail non monotone vs alpha.")
    _check(vL_list, +1, "VaR_left");  _check(eL_list, +1, "ES_left")
    _check(vR_list, -1, "VaR_right"); _check(eR_list, -1, "ES_right")

def _enforce_tail_monotonicity(vals: List[float], is_left: bool) -> List[float]:
    out = list(vals); n = len(out)
    if is_left:
        for k in range(1, n):
            if not np.isfinite(out[k]): out[k] = out[k-1]
            if out[k] < out[k-1]: out[k] = out[k-1]
        out = [min(0.0, x if np.isfinite(x) else 0.0) for x in out]
    else:
        for k in range(1, n):
            if not np.isfinite(out[k]): out[k] = out[k-1]
            if out[k] > out[k-1]: out[k] = out[k-1]
        out = [max(0.0, x if np.isfinite(x) else 0.0) for x in out]
    return out

def _enforce_pairwise_es_vs_var(row_V: List[float], row_E: List[float], is_left: bool) -> Tuple[List[float], List[float]]:
    V, E = list(row_V), list(row_E)
    for k in range(len(V)):
        v = 0.0 if not np.isfinite(V[k]) else V[k]
        e = 0.0 if not np.isfinite(E[k]) else E[k]
        if is_left:
            v = min(0.0, v); e = min(0.0, e)
            if e > v: e = v
        else:
            v = max(0.0, v); e = max(0.0, e)
            if e < v: e = v
        V[k], E[k] = v, e
    return V, E

def sanity_validate_sheet(df: pd.DataFrame, label: str) -> None:
    num = df.select_dtypes(include=[np.number])
    alpha_cols = [c for c in num.columns if c.startswith("alpha_")]
    core = num.drop(columns=alpha_cols, errors="ignore")
    if core.isna().sum().sum() != 0:
        nan_cols = core.columns[core.isna().any()].tolist()
        print(f"WARN [{label}] NaN detected in numeric columns: {nan_cols}", flush=True)
    sfxs = [f"a{str(a).replace('0.','0p')}" for a in sorted(ALPHAS)]
    for s in sfxs:
        if f"VaR_left_{s}" in df and f"ES_left_{s}" in df:
            if not ((df[f"VaR_left_{s}"] <= EPS).all() and (df[f"ES_left_{s}"] <= EPS).all()):
                raise AssertionError(f"[{label}] left signs invalid {s}")
            if not ((df[f"ES_left_{s}"] <= df[f"VaR_left_{s}"] + 1e-9).all()):
                raise AssertionError(f"[{label}] ES/VaR ordering invalid (left) {s}")
        if f"VaR_right_{s}" in df and f"ES_right_{s}" in df:
            if not ((df[f"VaR_right_{s}"] >= -EPS).all() and (df[f"ES_right_{s}"] >= -EPS).all()):
                raise AssertionError(f"[{label}] right signs invalid {s}")
            if not ((df[f"ES_right_{s}"] >= df[f"VaR_right_{s}"] - 1e-9).all()):
                raise AssertionError(f"[{label}] ES/VaR ordering invalid (right) {s}")

def sanity_validate_regime(df1: pd.DataFrame) -> None:
    if "p_gas" in df1:
        if not df1["p_gas"].between(0.0-EPS, 1.0+EPS).all():
            raise AssertionError("[1Y/5m] p_gas not in [0,1]..")
    if "regime" in df1:
        if not df1["regime"].dropna().isin(["GAS","FLUIDE"]).all():
            raise AssertionError("[1Y/5m] regime invalid.")

def smooth_evt_params(evt_df: pd.DataFrame, halflife: int = 10) -> pd.DataFrame:
    if evt_df.empty: return evt_df
    evt = evt_df.copy()
    for side in ("left","right"):
        for par in ("u","xi","beta"):
            c = f"EVT_{side}_{par}"
            if c in evt.columns:
                evt[c] = pd.to_numeric(evt[c], errors="coerce").ewm(halflife=halflife, min_periods=1).mean()
    return evt

def dynamic_thresholds_for_date(reg: str) -> Tuple[float,float]:
    if reg == "GAS":
        return EVT_XI_USE_MAX_GAS, MAG_CAP_GAS
    else:
        return EVT_XI_USE_MAX_FLU, MAG_CAP_FLU

def _gpu_sort_cumsum(X: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    if not USE_GPU or cp is None:
        Sx = np.sort(X, axis=1); cSx = np.cumsum(Sx, axis=1); return Sx, cSx
    try:
        Xg = cp.asarray(X)
        Sxg = cp.sort(Xg, axis=1)
        cSxg = cp.cumsum(Sxg, axis=1)
        Sx = cp.asnumpy(Sxg); cSx = cp.asnumpy(cSxg)
        del Xg, Sxg, cSxg
        cp._default_memory_pool.free_all_blocks()
        return Sx, cSx
    except Exception:
        Sx = np.sort(X, axis=1); cSx = np.cumsum(Sx, axis=1); return Sx, cSx

def expanding_robust_scale(s: pd.Series, min_periods: int = 20) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce")
    med = x.expanding(min_periods=min_periods).median()
    mad = (x - med).abs().expanding(min_periods=min_periods).median() * 1.4826
    std = x.expanding(min_periods=min_periods).std(ddof=1)
    sc = mad.where((mad > EPS) & np.isfinite(mad), std)
    sc = sc.where((sc > EPS) & np.isfinite(sc), 1.0)
    return sc

def hybrid_var_es(r: pd.Series, win: int, alphas: List[float], evt_df: pd.DataFrame,
                  side: str, regime_hint: Optional[pd.Series] = None) -> Tuple[pd.DataFrame, pd.Series]:
    sorted_alphas = sorted(alphas)
    alpha_sfx = [f"a{str(a).replace('0.','0p')}" for a in sorted_alphas]
    out = pd.DataFrame(index=r.index)
    edf_frac = pd.Series(np.nan, index=r.index, dtype=float)
    if len(r) < win:
        return out, edf_frac

    arr = pd.to_numeric(r, errors="coerce").to_numpy(dtype=np.float64)
    X_all = win_view(arr, win)
    if X_all.size == 0 or X_all.shape[1] < win:
        return out, edf_frac
    idx = r.index[win-1:]
    Xhist = X_all[:, :-1]
    Sx, cSx = _gpu_sort_cumsum(Xhist)
    E_raw = evt_df.reindex(idx).shift(1)
    E = smooth_evt_params(E_raw)
    reg_hint = regime_hint.shift(1).reindex(idx) if regime_hint is not None else None

    def _sc_series(col: str) -> pd.Series:
        if col not in E.columns:
            return pd.Series(index=E.index, dtype=float)
        return expanding_robust_scale(E[col]).shift(1)
    sc_right_xi   = _sc_series("EVT_right_xi")
    sc_right_beta = _sc_series("EVT_right_beta")
    sc_right_u    = _sc_series("EVT_right_u")
    sc_left_xi    = _sc_series("EVT_left_xi")
    sc_left_beta  = _sc_series("EVT_left_beta")
    sc_left_u     = _sc_series("EVT_left_u")

    alpha_max = max(sorted_alphas)
    CONT_SIGMA = 6.0

    for i, t in enumerate(idx):
        reg = "GAS"
        if reg_hint is not None and pd.notna(reg_hint.loc[t]):
            reg = str(reg_hint.loc[t])
        xi_use_max, mag_cap = dynamic_thresholds_for_date(reg)

        row_V: List[float] = []
        row_E: List[float] = []
        use_evt = False
        cur_ret_prev = float(Xhist[i, -1]) if Xhist.shape[1] > 0 else np.nan
        mad_i = _mad_vec(Xhist[i, :])
        thr_i = mad_i if (np.isfinite(mad_i) and mad_i > 0) else float(np.nanstd(Xhist[i, :], ddof=1))
        shock = (np.isfinite(cur_ret_prev) and np.isfinite(thr_i) and abs(cur_ret_prev) > 3.0 * thr_i)

        if side == "right":
            cols = ["EVT_right_xi", "EVT_right_beta", "EVT_right_pu", "EVT_right_u"]
            params = E.loc[t, cols] if all(c in E.columns for c in cols) else pd.Series([np.nan]*4, index=cols)
            if not params.isna().any():
                xi, beta, pu, u = map(float, params.values)
                if abs(xi) < xi_use_max and beta > 0 and pu > alpha_max:
                    v_evt_min, _ = evt_quant_es(u, xi, beta, pu, min(sorted_alphas), "right")
                    v_edf_min = Sx[i, int(max(0, math.ceil((1.0 - min(sorted_alphas)) * (Xhist.shape[1])) - 1))]
                    if np.isfinite(v_evt_min) and np.isfinite(v_edf_min) and (v_evt_min <= mag_cap * max(v_edf_min, EPS)):
                        Y = Xhist[i, :] - u; Y = Y[Y > 0]
                        ksD = _ks_gpd_distance(Y, xi, beta)
                        ok_ks = bool(np.isfinite(ksD) and ksD <= (0.22 if reg == "FLUIDE" else 0.20))
                        ok_cont = True
                        if (not shock) and i > 0:
                            tprev = idx[i-1]
                            xi_p = float(E.loc[tprev, "EVT_right_xi"]) if "EVT_right_xi" in E.columns and pd.notna(E.loc[tprev, "EVT_right_xi"]) else np.nan
                            b_p  = float(E.loc[tprev, "EVT_right_beta"]) if "EVT_right_beta" in E.columns and pd.notna(E.loc[tprev, "EVT_right_beta"]) else np.nan
                            u_p  = float(E.loc[tprev, "EVT_right_u"]) if "EVT_right_u" in E.columns and pd.notna(E.loc[tprev, "EVT_right_u"]) else np.nan
                            sc_xi = float(sc_right_xi.get(t, np.nan));   sc_xi = sc_xi if (np.isfinite(sc_xi) and sc_xi > EPS) else 1.0
                            sc_be = float(sc_right_beta.get(t, np.nan)); sc_be = sc_be if (np.isfinite(sc_be) and sc_be > EPS) else 1.0
                            sc_u  = float(sc_right_u.get(t, np.nan));    sc_u  = sc_u  if (np.isfinite(sc_u)  and sc_u  > EPS) else 1.0
                            score = 0.0
                            if np.isfinite(xi_p): score = max(score, abs(xi - xi_p)/sc_xi)
                            if np.isfinite(b_p):  score = max(score, abs(beta - b_p)/sc_be)
                            if np.isfinite(u_p):  score = max(score, abs(u - u_p)/sc_u)
                            if score > CONT_SIGMA: ok_cont = False
                        use_evt = (ok_ks and ok_cont)

            for a in sorted_alphas:
                if use_evt:
                    v, e = evt_quant_es(u, xi, beta, pu, a, "right")
                else:
                    vL_b, eL_b, vR_b, eR_b = edf_var_es_from_sorted(Sx, cSx, a)
                    v, e = vR_b[i], eR_b[i]
                row_V.append(float(v)); row_E.append(float(e))

            row_V, row_E = _enforce_pairwise_es_vs_var(row_V, row_E, is_left=False)
            row_V = _enforce_tail_monotonicity(row_V, is_left=False)
            row_E = _enforce_tail_monotonicity(row_E, is_left=False)

        else:
            cols = ["EVT_left_xi", "EVT_left_beta", "EVT_left_pu", "EVT_left_u"]
            params = E.loc[t, cols] if all(c in E.columns for c in cols) else pd.Series([np.nan]*4, index=cols)
            if not params.isna().any():
                xi, beta, pu, u = map(float, params.values)
                if abs(xi) < xi_use_max and beta > 0 and pu > alpha_max:
                    v_evt_min, _ = evt_quant_es(u, xi, beta, pu, min(sorted_alphas), "left")
                    v_edf_min = Sx[i, int(max(0, math.ceil(min(sorted_alphas) * (Xhist.shape[1])) - 1))]
                    if np.isfinite(v_evt_min) and np.isfinite(v_edf_min) and (abs(v_evt_min) <= mag_cap * max(abs(v_edf_min), EPS)):
                        Y = -Xhist[i, :] - u; Y = Y[Y > 0]
                        ksD = _ks_gpd_distance(Y, xi, beta)
                        ok_ks = bool(np.isfinite(ksD) and ksD <= (0.18 if reg == "FLUIDE" else 0.15))
                        ok_cont = True
                        if (not shock) and i > 0:
                            tprev = idx[i-1]
                            xi_p = float(E.loc[tprev, "EVT_left_xi"]) if "EVT_left_xi" in E.columns and pd.notna(E.loc[tprev, "EVT_left_xi"]) else np.nan
                            b_p  = float(E.loc[tprev, "EVT_left_beta"]) if "EVT_left_beta" in E.columns and pd.notna(E.loc[tprev, "EVT_left_beta"]) else np.nan
                            u_p  = float(E.loc[tprev, "EVT_left_u"]) if "EVT_left_u" in E.columns and pd.notna(E.loc[tprev, "EVT_left_u"]) else np.nan
                            sc_xi = float(sc_left_xi.get(t, np.nan));    sc_xi = sc_xi if (np.isfinite(sc_xi) and sc_xi > EPS) else 1.0
                            sc_be = float(sc_left_beta.get(t, np.nan));  sc_be = sc_be if (np.isfinite(sc_be) and sc_be > EPS) else 1.0
                            sc_u  = float(sc_left_u.get(t, np.nan));     sc_u  = sc_u  if (np.isfinite(sc_u)  and sc_u  > EPS) else 1.0
                            score = 0.0
                            if np.isfinite(xi_p): score = max(score, abs(xi - xi_p)/sc_xi)
                            if np.isfinite(b_p):  score = max(score, abs(beta - b_p)/sc_be)
                            if np.isfinite(u_p):  score = max(score, abs(u - u_p)/sc_u)
                            if score > CONT_SIGMA: ok_cont = False
                        use_evt = (ok_ks and ok_cont)

            for a in sorted_alphas:
                if use_evt:
                    v, e = evt_quant_es(u, xi, beta, pu, a, "left")
                else:
                    vL_b, eL_b, vR_b, eR_b = edf_var_es_from_sorted(Sx, cSx, a)
                    v, e = vL_b[i], eL_b[i]
                row_V.append(float(v)); row_E.append(float(e))

            row_V, row_E = _enforce_pairwise_es_vs_var(row_V, row_E, is_left=True)
            row_V = _enforce_tail_monotonicity(row_V, is_left=True)
            row_E = _enforce_tail_monotonicity(row_E, is_left=True)

        for k, s in enumerate(alpha_sfx):
            out.loc[t, f"VaR_{side}_{s}"] = row_V[k]
            out.loc[t, f"ES_{side}_{s}"]  = row_E[k]
        edf_frac.loc[t] = 0.0 if use_evt else 1.0

    return out, edf_frac

def ensure_alpha_cols(df: pd.DataFrame, label: str) -> None:
    for a, s in zip(ALPHAS, ALPHA_SFX):
        EL = df.get(f"ES_left_{s}", None)
        ER = df.get(f"ES_right_{s}", None)
        mu = df.get("mu", None)
        if EL is not None and f"alpha_minus_{s}" not in df.columns:
            df[f"alpha_minus_{s}"] = pd.to_numeric(EL, errors="coerce")
        if ER is not None and f"alpha_plus_{s}" not in df.columns:
            df[f"alpha_plus_{s}"] = pd.to_numeric(ER, errors="coerce")
        if (mu is not None) and (EL is not None) and (ER is not None) and f"alpha_mu_{s}" not in df.columns:
            ELp = pd.to_numeric(EL, errors="coerce")
            ERp = pd.to_numeric(ER, errors="coerce")
            mup = pd.to_numeric(mu, errors="coerce")
            df[f"alpha_mu_{s}"] = (mup - a * (ELp + ERp)) / max(1e-12, (1.0 - 2.0 * a))

def _fetch_daily_one(sym: str) -> Tuple[str, Optional[str], Optional[str]]:
    try:
        df = get_daily_data(sym, START_DATE.isoformat(), END_DATE.isoformat())
        path = save_parquet_safe(df, daily_path(sym))
        return sym, "OK", str(path)
    except Exception as e:
        return sym, f"ERR {type(e).__name__}: {e}", None

def _fetch_5m_one_dynamic(sym: str) -> Tuple[str, Optional[str], Optional[str]]:
    try:
        df = download_max_5m_history(sym, END_DATE, max_probe_days=MAX_PROBE_5M_DAYS, chunk_days=FALLBACK_INTRADAY_LIMIT_DAYS)
        path = save_parquet_safe(df, intraday_path(sym))
        return sym, f"OK {len(df):,} rows", str(path)
    except Exception as e:
        return sym, f"ERR {type(e).__name__}: {e}", None

def download_all(tickers: List[str]) -> None:
    print(f"[DL] Phase 1/2: DAILY x{len(tickers)} | GPU={'ON' if USE_GPU else 'OFF'}")
    with ThreadPoolExecutor(max_workers=DL_MAX_WORKERS) as pool:
        futs = [pool.submit(_fetch_daily_one, s) for s in tickers]
        for f in tqdm(as_completed(futs), total=len(futs), desc="daily", leave=False):
            sym, status, path = f.result()
            print(f"  - [{sym}] daily -> {status} {('('+path+')' if path else '')}", flush=True)

    print("[DL] Pause 2.2s avant intraday 5m…", flush=True)
    pytime.sleep(2.2)

    print(f"[DL] Phase 2/2: INTRADAY 5m dynamique x{len(tickers)} (probe≤{MAX_PROBE_5M_DAYS}j, chunk={FALLBACK_INTRADAY_LIMIT_DAYS}j)")
    with ThreadPoolExecutor(max_workers=DL_MAX_WORKERS) as pool:
        futs = [pool.submit(_fetch_5m_one_dynamic, s) for s in tickers]
        for f in tqdm(as_completed(futs), total=len(futs), desc="5m", leave=False):
            sym, status, path = f.result()
            print(f"  - [{sym}] 5m -> {status} {('('+path+')' if path else '')}", flush=True)

def pit_only_cols(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    drop_patterns = (r'.*_raw$', r'^(skew|kurt_excess)$')
    to_drop = [c for c in df.columns if any(re.match(p, c) for p in drop_patterns)]
    return df.drop(columns=to_drop, errors="ignore")

def build_M_pit_from_returns(
    ret: pd.Series,
    ohlc: pd.DataFrame,
    win: int,
    sigma_halflife: Optional[int] = None,
    gk_weight: float = 0.7
) -> pd.DataFrame:
    M_core = gaussian_roll(ret, win)
    if sigma_halflife is None:
        sigma_halflife = max(10, win // 8)
    sigma_gk = sigma_diffusion_from_ohlc(ohlc, win, halflife=sigma_halflife)

    M = M_core.copy()
    M["mu_raw"]    = M_core["mu"]
    M["sigma_raw"] = M_core["sigma"]
    M["mu"]        = M_core["mu"].shift(1)

    sig_rolled_pit = M_core["sigma"].shift(1)
    sig_gk_pit     = sigma_gk.reindex(M.index)
    sig_gk_pit = sig_gk_pit.where(sig_gk_pit.notna(), sig_rolled_pit)

    M["sigma"] = (gk_weight * sig_gk_pit + (1.0 - gk_weight) * sig_rolled_pit)
    return M

def save_stage1_symbol(sym: str,
                       daily1: pd.DataFrame,
                       daily0: pd.DataFrame,
                       intraday5m: pd.DataFrame,
                       daily5m: pd.DataFrame,
                       daily_from_mean: pd.DataFrame,
                       ohlc_10y: pd.DataFrame,
                       ohlc_1y: pd.DataFrame,
                       ohlc_5m_full: pd.DataFrame,
                       ohlcA_5m: pd.DataFrame,
                       ohlcB_5m: pd.DataFrame) -> Dict[str, str]:
    paths = {}
    if daily1 is not None and not daily1.empty:
        paths["Daily_1Y"] = str(save_parquet_safe(daily1, stage1_path(sym, "Daily_1Y")))
    if daily0 is not None and not daily0.empty:
        paths["Daily_10Y"] = str(save_parquet_safe(daily0, stage1_path(sym, "Daily_10Y")))
    if intraday5m is not None and not intraday5m.empty:
        paths["Intraday_5m"] = str(save_parquet_safe(intraday5m, stage1_path(sym, "Intraday_5m")))
    if daily5m is not None and not daily5m.empty:
        paths["Daily_5m"] = str(save_parquet_safe(daily5m, stage1_path(sym, "Daily_5m")))
    if daily_from_mean is not None and not daily_from_mean.empty:
        paths["Daily_fromMean_5m"] = str(save_parquet_safe(daily_from_mean, stage1_path(sym, "Daily_fromMean_5m")))
    if ohlc_10y is not None and not ohlc_10y.empty:
        paths["OHLC_10Y"] = str(save_parquet_safe(ohlc_10y.reset_index(), stage1_path(sym, "OHLC_10Y")))
    if ohlc_1y is not None and not ohlc_1y.empty:
        paths["OHLC_1Y"] = str(save_parquet_safe(ohlc_1y.reset_index(), stage1_path(sym, "OHLC_1Y")))
    if ohlc_5m_full is not None and not ohlc_5m_full.empty:
        paths["OHLC_5m"] = str(save_parquet_safe(ohlc_5m_full.reset_index(), stage1_path(sym, "OHLC_5m_raw")))
    if ohlcA_5m is not None and not ohlcA_5m.empty:
        paths["OHLC_5m_A"] = str(save_parquet_safe(ohlcA_5m.reset_index(), stage1_path(sym, "OHLC_5m_streamA")))
    if ohlcB_5m is not None and not ohlcB_5m.empty:
        paths["OHLC_5m_B"] = str(save_parquet_safe(ohlcB_5m.reset_index(), stage1_path(sym, "OHLC_5m_streamB")))
    manifest = {
        "symbol": sym,
        "as_of": END_DATE.isoformat(),
        "paths": paths,
        "notes": "Stage1 prêt pour Cellule 2. 5m: split 1/2 (A=moyenne, B=calculs quotidiens depuis moyenne). Aucun fichier Excel."
    }
    with open(stage1_json_path(sym, "manifest"), "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    return paths

def compute_symbol(sym: str) -> Optional[str]:
    t0 = pytime.time()
    try:
        ohlc = trim_to_asof(load_parquet_safe(daily_path(sym)), END_DATE)
        ret_raw = log_returns(ohlc)
        if len(ret_raw) < WIN_1Y + 20:
            print(f"[{sym}] Insufficient daily history (<{WIN_1Y+20}). Skip.", flush=True)
            return None

        print(f"[{sym}] 1/4  Calculs 1Y (daily)…", flush=True)
        M1   = build_M_pit_from_returns(ret_raw, ohlc, WIN_1Y)
        EVT1 = evt_params_from_series(ret_raw, WIN_1Y)
        H1   = rolling_hurst(ret_raw, WIN_1Y)
        L1   = rolling_lyap_kernel(ret_raw, WIN_1Y, m=KR_DIM, tau=KR_TAU)
        ENT1 = entropy_mixture_series(ret_raw, WIN_1Y, EVT1)
        REG1 = classify_gas_fluid_linear_nonlin(ret_raw, WIN_1Y, p_ar=6, lam=KR_RIDGE, kernel=KR_KERNEL)
        PRE1 = premice_signals(M1, EVT1, H1, L1, win_long=WIN_1Y, win_short=63)
        Q1_R, _ = hybrid_var_es(ret_raw, WIN_1Y, ALPHAS, EVT1, side="right", regime_hint=REG1.get("regime"))
        Q1_L, _ = hybrid_var_es(ret_raw, WIN_1Y, ALPHAS, EVT1, side="left",  regime_hint=REG1.get("regime"))
        daily1 = (M1.join(EVT1, how='inner').join(Q1_R, how='inner').join(Q1_L, how='inner'))
        daily1[H1.name] = H1
        daily1["Lyap_KR_W252"] = L1
        daily1["Lyap_AR6_W252"] = L1
        daily1 = daily1.join(REG1).join(PRE1).join(ENT1.add_prefix("ENT_"))
        evt_cols = [c for c in daily1.columns if c.startswith("EVT_")]
        if evt_cols:
            daily1 = daily1.rename(columns={c: c + "_raw" for c in evt_cols})
            daily1 = daily1.join(
                daily1[[c + "_raw" for c in evt_cols]]
                .rename(columns=lambda c: c.replace("_raw",""))
                .shift(1)
                .add_suffix("_pit")
            )
        if "Hurst_W252" in daily1.columns:
            daily1["Hurst_W252"] = daily1["Hurst_W252"].shift(1)
        ensure_alpha_cols(daily1, "1Y")
        daily1_valid = daily1.dropna(how="any", subset=["mu", "sigma"] + [f"VaR_left_{ALPHA_SFX[0]}"])
        sanity_validate_monotonicity(daily1_valid, "1Y")
        sanity_validate_entropy(daily1_valid, "1Y")
        sanity_validate_sheet(daily1_valid, "1Y")
        sanity_validate_regime(daily1_valid)
        _assert_pit_alignment(daily1, ["mu","sigma"], "Daily_1Y")
        daily1_export = pit_only_cols(daily1).iloc[-WIN_1Y:].copy()
        daily0_export = pd.DataFrame()
        if len(ret_raw) > WIN_10Y + 10:
            print(f"[{sym}] 2/4  Calculs 10Y (daily)…", flush=True)
            M0   = build_M_pit_from_returns(ret_raw, ohlc, WIN_10Y)
            EVT0 = evt_params_from_series(ret_raw, WIN_10Y)
            Q0_R, _ = hybrid_var_es(ret_raw, WIN_10Y, ALPHAS, EVT0, side="right")
            Q0_L, _ = hybrid_var_es(ret_raw, WIN_10Y, ALPHAS, EVT0, side="left")
            ENT0 = entropy_mixture_series(ret_raw, WIN_10Y, EVT0)
            daily0 = M0.join(EVT0, how='inner').join(Q0_R, how='inner').join(Q0_L, how='inner')
            daily0 = daily0.join(ENT0.add_prefix("ENT_"))
            evt0_cols = [c for c in daily0.columns if c.startswith("EVT_")]
            if evt0_cols:
                daily0 = daily0.rename(columns={c: c + "_raw" for c in evt0_cols})
                daily0 = daily0.join(
                    daily0[[c + "_raw" for c in evt0_cols]]
                    .rename(columns=lambda c: c.replace("_raw",""))
                    .shift(1)
                    .add_suffix("_pit")
                )
            ensure_alpha_cols(daily0, "10Y")
            daily0_valid = daily0.dropna(how="any", subset=["mu", "sigma"] + [f"VaR_left_{ALPHA_SFX[0]}"])
            if not daily0_valid.empty:
                sanity_validate_monotonicity(daily0_valid, "10Y")
                sanity_validate_entropy(daily0_valid, "10Y")
                sanity_validate_sheet(daily0_valid, "10Y")
                daily0_export = pit_only_cols(daily0.loc[daily0.index >= CUT_10Y]).copy()
        else:
            print(f"[{sym}] 2/4  Not enough for 10Y. Skip.", flush=True)
        print(f"[{sym}] 3/4  Chargement intraday 5m…", flush=True)
        ohlc_5m_full = trim_to_asof(load_parquet_safe(intraday_path(sym)), END_DATE)
        ret_5m  = log_returns(ohlc_5m_full, price_col="close") if not ohlc_5m_full.empty else pd.Series(dtype=float)
        BPD, WIN_5M = compute_win_5m_pit(ohlc_5m_full, PREWARM_5M_DAYS, min_win=64)
        daily5m_export  = pd.DataFrame()
        intraday5m_export = pd.DataFrame()
        if WIN_5M >= 64 and len(ret_5m) > WIN_5M:
            print(f"[{sym}] 4/4  Calculs 5m (WIN_5M={WIN_5M}, BPD≈{BPD}) & split 1/2…", flush=True)
            M5   = build_M_pit_from_returns(ret_5m, ohlc_5m_full, WIN_5M, sigma_halflife=max(10, WIN_5M//6))
            EVT5 = evt_params_from_series(ret_5m, WIN_5M)
            H5   = rolling_hurst(ret_5m, WIN_5M)
            L5   = rolling_lyap_kernel(ret_5m, WIN_5M, m=KR_DIM, tau=KR_TAU)
            ENT5 = entropy_mixture_series(ret_5m, WIN_5M, EVT5, xi_shrink=0.9*EVT_XI_USE_MAX_GAS)
            REG5 = classify_gas_fluid_linear_nonlin(ret_5m, WIN_5M, p_ar=6, lam=KR_RIDGE, kernel=KR_KERNEL)
            PRE5 = premice_signals(M5, EVT5, H5, L5, win_long=5*WIN_5M, win_short=WIN_5M)
            Q5_R, _ = hybrid_var_es(ret_5m, WIN_5M, ALPHAS, EVT5, side="right", regime_hint=REG5.get("regime"))
            Q5_L, _ = hybrid_var_es(ret_5m, WIN_5M, ALPHAS, EVT5, side="left",  regime_hint=REG5.get("regime"))
            tab5 = (M5.join(EVT5, how="left").join(Q5_R, how="left").join(Q5_L, how="left"))
            tab5["Hurst_W5m"]     = H5.reindex(tab5.index)
            tab5["Lyap_KR_W5m"]   = L5.reindex(tab5.index)
            tab5["Lyap_AR6_W5m"]  = tab5["Lyap_KR_W5m"]
            tab5 = tab5.join(REG5).join(PRE5).join(ENT5.add_prefix("ENT_"))
            evt5_cols = [c for c in tab5.columns if c.startswith("EVT_")]
            if evt5_cols:
                tab5 = tab5.rename(columns={c: c + "_raw" for c in evt5_cols})
                tab5 = tab5.join(
                    tab5[[c + "_raw" for c in evt5_cols]]
                    .rename(columns=lambda c: c.replace("_raw",""))
                    .shift(1)
                    .add_suffix("_pit")
                )
            if "Hurst_W5m" in tab5.columns:
                tab5["Hurst_W5m"] = tab5["Hurst_W5m"].shift(1)

            ensure_alpha_cols(tab5, "5m")
            tab5 = tab5.sort_index()

            try:
                tab5_valid = tab5.dropna(subset=[col for col in ["mu","sigma"] if col in tab5.columns])
                if not tab5_valid.empty:
                    sanity_validate_monotonicity(tab5_valid, "5m")
                    sanity_validate_entropy(tab5_valid, "5m")
                    sanity_validate_sheet(tab5_valid, "5m")
                    sanity_validate_regime(tab5_valid)
            except AssertionError as e:
                print(f"[{sym}] WARN validate 5m: {e}", flush=True)

            _assert_pit_alignment(tab5, ["mu","sigma"], "Intraday_5m")
            intraday_cut = END_TS - pd.Timedelta(days=DAILY_5M_DAYS-1)
            intraday5m_export = pit_only_cols(tab5.loc[tab5.index >= intraday_cut]).copy()
            if not intraday5m_export.empty:
                daily5m_export = intraday5m_export.groupby(intraday5m_export.index.floor('D')).last()
                daily5m_export = pit_only_cols(daily5m_export)
        else:
            print(f"[{sym}] 4/4  5m insuffisant (WIN_5M={WIN_5M}, n={len(ret_5m)})", flush=True)

        ohlcA_5m, ohlcB_5m, daily_from_mean = split_5m_make_daily_from_mean(ohlc_5m_full)
        ohlc_10y = ohlc.loc[ohlc.index >= CUT_10Y]
        ohlc_1y  = ohlc.loc[ohlc.index >= CUT_1Y]
        daily1_export = daily1_export.drop(columns=["kurt_exces"], errors="ignore")
        if not daily0_export.empty:
            daily0_export = daily0_export.drop(columns=["kurt_exces"], errors="ignore")
        if not intraday5m_export.empty:
            intraday5m_export = intraday5m_export.drop(columns=["kurt_exces"], errors="ignore")
        if not daily5m_export.empty:
            daily5m_export = daily5m_export.drop(columns=["kurt_exces"], errors="ignore")

        paths = save_stage1_symbol(
            sym=sym,
            daily1=daily1_export,
            daily0=daily0_export,
            intraday5m=intraday5m_export,
            daily5m=daily5m_export,
            daily_from_mean=daily_from_mean,
            ohlc_10y=ohlc_10y,
            ohlc_1y=ohlc_1y,
            ohlc_5m_full=ohlc_5m_full,
            ohlcA_5m=ohlcA_5m,
            ohlcB_5m=ohlcB_5m
        )

        dt = pytime.time() - t0
        print(f"[{sym}] OK | Stage1 saved ({len(paths)} files) | GPU={'ON' if USE_GPU else 'OFF'} | {dt:.2f}s", flush=True)
        return json.dumps(paths)

    except Exception as ex:
        print(f"[{sym}] ERROR: {type(ex).__name__}: {ex}", flush=True)
        traceback.print_exc()
        return None
    finally:
        gc.collect()
        if USE_GPU and cp is not None:
            try:
                cp._default_memory_pool.free_all_blocks()
            except Exception:
                pass

def compute_all(tickers: List[str]) -> None:
    print(f"[RUN] Compute x{len(tickers)} | workers={COMPUTE_MAX_WORKERS} | GPU={'ON' if USE_GPU else 'OFF'}", flush=True)
    with ProcessPoolExecutor(max_workers=COMPUTE_MAX_WORKERS) as pool:
        futs = {pool.submit(compute_symbol, s): s for s in tickers}
        for f in tqdm(as_completed(futs), total=len(futs), desc="compute", leave=False):
            sym = futs[f]
            try:
                res = f.result(timeout=3600)
                print(f"  - [{sym}] done -> {('OK' if res else 'None')}", flush=True)
            except Exception as ex:
                print(f"  - [{sym}] FAILED: {type(ex).__name__}: {ex}", flush=True)

def main():
    print(f"[INIT] CORE PHILO (PATCHED • NO-PROXY) | {len(TICKERS)} tickers | {START_DATE.isoformat()}→{END_DATE.isoformat()}")
    print(f"[HW]   GPU={'ON' if USE_GPU else 'OFF'} (CuPy={'yes' if USE_GPU else 'no'})  CPUs for compute={COMPUTE_MAX_WORKERS}", flush=True)
    download_all(TICKERS)
    compute_all(TICKERS)

if __name__ == "__main__":
    main()


[INIT] CORE PHILO (PATCHED • NO-PROXY) | 8 tickers | 2011-10-14→2026-10-10
[HW]   GPU=ON (CuPy=yes)  CPUs for compute=2
[DL] Phase 1/2: DAILY x8 | GPU=ON


daily:   0%|          | 0/8 [00:00<?, ?it/s]

  - [usdchf.forex] daily -> OK (data_raw/USDCHF_FOREX_daily.parquet)


daily:  12%|█▎        | 1/8 [00:00<00:03,  2.09it/s]

  - [usdjpy.forex] daily -> OK (data_raw/USDJPY_FOREX_daily.parquet)
  - [gbpchf.forex] daily -> OK (data_raw/GBPCHF_FOREX_daily.parquet)


...

5m:  88%|████████▊ | 7/8 [00:35<00:01,  1.64s/it]

  - [usdcad.forex] 5m -> OK 37,140 rows (data_raw/USDCAD_FOREX_5m.parquet)


[RUN] Compute x8 | workers=2 | GPU=ON


compute:   0%|          | 0/8 [00:00<?, ?it/s]

[usdchf.forex] 1/4  Calculs 1Y (daily)…[usdjpy.forex] 1/4  Calculs 1Y (daily)…

[usdchf.forex] 2/4  Calculs 10Y (daily)…
[usdjpy.forex] 2/4  Calculs 10Y (daily)…
[usdjpy.forex] 3/4  Chargement intraday 5m…
[usdjpy.forex] 4/4  Calculs 5m (WIN_5M=1440, BPD≈288) & split 1/2…
[usdchf.forex] 3/4  Chargement intraday 5m…
[usdchf.forex] 4/4  Calculs 5m (WIN_5M=1440, BPD≈288) & split 1/2…


In [ ]:
SER_MATRICES = {
"A1_AUDIT_V1": np.array([
    [0.8842,0.0411,0.0005,0.0002,0.0015,0.0018,0.0405,0.0004,0.0002,0.0016,0.0019,0.0125,0.0018,0.0001,0.0001,0.0005,0.0008,0.0019,0.0001,0.0001,0.0006,0.0009],
    [0.2985,0.2550,0.6215,0.0010,0.0025,0.0015,0.0012,0.0001,0.0001,0.0001,0.0001,0.0080,0.0050,0.0025,0.0001,0.0005,0.0003,0.0002,0.0001,0.0001,0.0001,0.0001],
    [0.0512,0.0015,0.7988,0.1150,0.0155,0.0025,0.0010,0.0005,0.0001,0.0001,0.0001,0.0050,0.0005,0.0035,0.0015,0.0010,0.0005,0.0002,0.0001,0.0001,0.0001,0.0001],
    [0.0115,0.0005,0.0025,0.8950,0.0750,0.0050,0.0005,0.0002,0.0001,0.0001,0.0001,0.0025,0.0002,0.0005,0.0025,0.0020,0.0010,0.0001,0.0001,0.0001,0.0001,0.0001],
    [0.1550,0.0010,0.0005,0.0002,0.2650,0.5510,0.0025,0.0005,0.0001,0.0001,0.0001,0.0110,0.0005,0.0002,0.0001,0.0025,0.0050,0.0020,0.0002,0.0001,0.0001,0.0001],
    [0.6510,0.0015,0.0005,0.0001,0.0001,0.2110,0.1050,0.0025,0.0005,0.0002,0.0001,0.0150,0.0005,0.0002,0.0001,0.0005,0.0010,0.0050,0.0005,0.0002,0.0002,0.0001],
    [0.2995,0.0011,0.0001,0.0001,0.0001,0.0001,0.2535,0.6205,0.0010,0.0025,0.0015,0.0085,0.0001,0.0001,0.0001,0.0001,0.0002,0.0055,0.0025,0.0001,0.0005,0.0003],
    [0.0521,0.0005,0.0001,0.0001,0.0001,0.0001,0.0015,0.7950,0.1180,0.0150,0.0020,0.0055,0.0001,0.0001,0.0001,0.0001,0.0002,0.0005,0.0035,0.0015,0.0010,0.0005],
    [0.0122,0.0002,0.0001,0.0001,0.0001,0.0001,0.0005,0.0025,0.8925,0.0760,0.0052,0.0028,0.0001,0.0001,0.0001,0.0001,0.0002,0.0002,0.0005,0.0025,0.0020,0.0010],
    [0.1540,0.0001,0.0001,0.0001,0.0001,0.0001,0.0010,0.0005,0.0002,0.2680,0.5480,0.0125,0.0001,0.0001,0.0001,0.0001,0.0002,0.0005,0.0002,0.0001,0.0025,0.0050],
    [0.6525,0.0005,0.0002,0.0001,0.0001,0.0001,0.0015,0.0005,0.0002,0.0001,0.2080,0.1250,0.0002,0.0001,0.0001,0.0001,0.0002,0.0010,0.0005,0.0002,0.0005,0.0010],
    [0.0250,0.0025,0.0002,0.0001,0.0005,0.0008,0.0024,0.0002,0.0001,0.0006,0.0009,0.8210,0.0550,0.0010,0.0002,0.0025,0.0035,0.0540,0.0008,0.0002,0.0028,0.0035],
    [0.0150,0.0055,0.0035,0.0002,0.0005,0.0003,0.0002,0.0001,0.0001,0.0001,0.0001,0.3150,0.2110,0.6155,0.0015,0.0025,0.0010,0.0005,0.0001,0.0001,0.0001,0.0001],
    [0.0085,0.0005,0.0015,0.0005,0.0010,0.0005,0.0002,0.0001,0.0001,0.0001,0.0001,0.0650,0.0025,0.7550,0.1250,0.0250,0.0055,0.0005,0.0002,0.0001,0.0001,0.0001],
    [0.0035,0.0002,0.0002,0.0010,0.0015,0.0010,0.0001,0.0001,0.0001,0.0001,0.0001,0.0210,0.0005,0.0035,0.8511,0.0950,0.0150,0.0002,0.0001,0.0001,0.0001,0.0001],
    [0.0055,0.0002,0.0001,0.0001,0.0001,0.0002,0.0002,0.0001,0.0001,0.0001,0.0001,0.1850,0.0015,0.0005,0.0002,0.2250,0.5255,0.0250,0.0005,0.0002,0.0002,0.0002],
    [0.0850,0.0010,0.0005,0.0002,0.0005,0.0010,0.0008,0.0002,0.0001,0.0001,0.0001,0.5510,0.0025,0.0010,0.0005,0.0015,0.1880,0.1550,0.0025,0.0005,0.0005,0.0005],
    [0.0145,0.0001,0.0001,0.0001,0.0001,0.0002,0.0050,0.0035,0.0001,0.0005,0.0003,0.3125,0.0001,0.0001,0.0001,0.0002,0.0005,0.2150,0.6185,0.0015,0.0025,0.0010],
    [0.0088,0.0001,0.0001,0.0001,0.0001,0.0002,0.0005,0.0015,0.0005,0.0010,0.0005,0.0645,0.0001,0.0001,0.0001,0.0002,0.0005,0.0025,0.7580,0.1220,0.0240,0.0050],
    [0.0038,0.0001,0.0001,0.0001,0.0001,0.0001,0.0002,0.0005,0.0025,0.0020,0.0010,0.0215,0.0001,0.0001,0.0001,0.0001,0.0002,0.0005,0.0035,0.8511,0.0950,0.0150],
    [0.0058,0.0001,0.0001,0.0001,0.0001,0.0001,0.0002,0.0002,0.0001,0.0002,0.0002,0.1880,0.0002,0.0001,0.0001,0.0002,0.0005,0.0015,0.0005,0.0002,0.2811,0.6505],
    [0.0880,0.0005,0.0002,0.0001,0.0001,0.0002,0.0055,0.0025,0.0005,0.0005,0.0005,0.5480,0.0015,0.0005,0.0002,0.0005,0.0010,0.0025,0.0010,0.0005,0.0015,0.1988],
]),
"A1_AUDIT_V2": np.array([
    [0.6512,0.1235,0.0112,0.0015,0.0031,0.0041,0.1241,0.0109,0.0014,0.0029,0.0045,0.0315,0.0048,0.0005,0.0001,0.0008,0.0011,0.0051,0.0006,0.0001,0.0007,0.0012],
    [0.1845,0.4521,0.2188,0.0312,0.0115,0.0088,0.0101,0.0011,0.0002,0.0005,0.0009,0.0211,0.0399,0.0105,0.0015,0.0019,0.0012,0.0010,0.0001,0.0000,0.0001,0.0002],
    [0.0501,0.1033,0.5519,0.1884,0.0255,0.0101,0.0021,0.0003,0.0001,0.0001,0.0002,0.0050,0.0062,0.0217,0.0109,0.0028,0.0011,0.0001,0.0000,0.0000,0.0000,0.0000],
    [0.0113,0.0205,0.0876,0.6854,0.1211,0.0215,0.0008,0.0001,0.0000,0.0001,0.0001,0.0011,0.0019,0.0044,0.0201,0.0188,0.0033,0.0000,0.0000,0.0000,0.0000,0.0000],
    [0.0811,0.2541,0.0412,0.0118,0.3511,0.1569,0.0105,0.0011,0.0002,0.0003,0.0005,0.0255,0.0301,0.0021,0.0005,0.0199,0.0111,0.0009,0.0001,0.0000,0.0000,0.0001],
    [0.4812,0.0511,0.0055,0.0011,0.0889,0.1854,0.0498,0.0049,0.0009,0.0012,0.0019,0.0811,0.0088,0.0005,0.0001,0.0101,0.0251,0.0011,0.0002,0.0000,0.0001,0.0002],
    [0.1851,0.0105,0.0012,0.0002,0.0006,0.0011,0.4511,0.2199,0.0309,0.0118,0.0085,0.0215,0.0011,0.0001,0.0000,0.0001,0.0002,0.0405,0.0101,0.0014,0.0018,0.0012],
    [0.0505,0.0022,0.0003,0.0001,0.0001,0.0002,0.1041,0.5505,0.1891,0.0249,0.0105,0.0051,0.0002,0.0000,0.0000,0.0000,0.0000,0.0065,0.0211,0.0111,0.0029,0.0010],
    [0.0115,0.0008,0.0001,0.0000,0.0001,0.0001,0.0209,0.0881,0.6849,0.1215,0.0211,0.0012,0.0000,0.0000,0.0000,0.0000,0.0000,0.0021,0.0048,0.0199,0.0185,0.0034],
    [0.0815,0.0012,0.0002,0.0000,0.0001,0.0002,0.2551,0.0409,0.0115,0.3505,0.1571,0.0258,0.0001,0.0000,0.0000,0.0000,0.0001,0.0305,0.0022,0.0005,0.0201,0.0109],
    [0.4821,0.0051,0.0009,0.0001,0.0011,0.0018,0.0502,0.0051,0.0010,0.0891,0.1849,0.0815,0.0004,0.0000,0.0000,0.0001,0.0002,0.0091,0.0005,0.0001,0.0105,0.0249],
    [0.0411,0.0088,0.0008,0.0001,0.0015,0.0021,0.0091,0.0009,0.0001,0.0017,0.0025,0.5987,0.1301,0.0111,0.0015,0.0051,0.0088,0.1311,0.0109,0.0014,0.0055,0.0091],
    [0.0412,0.0611,0.0287,0.0031,0.0099,0.0051,0.0088,0.0008,0.0001,0.0009,0.0011,0.1503,0.4855,0.1511,0.0215,0.0101,0.0041,0.0051,0.0005,0.0001,0.0008,0.0009],
    [0.0111,0.0155,0.0899,0.0411,0.0188,0.0081,0.0011,0.0001,0.0000,0.0001,0.0002,0.0488,0.1011,0.4518,0.1551,0.0351,0.0101,0.0008,0.0001,0.0000,0.0001,0.0001],
    [0.0021,0.0031,0.0155,0.1511,0.0855,0.0199,0.0002,0.0000,0.0000,0.0000,0.0001,0.0101,0.0251,0.1011,0.4811,0.0915,0.0111,0.0001,0.0000,0.0000,0.0000,0.0000],
    [0.0511,0.0411,0.0099,0.0015,0.1511,0.0811,0.0055,0.0005,0.0001,0.0006,0.0008,0.2511,0.1811,0.0311,0.0051,0.1501,0.0351,0.0011,0.0001,0.0000,0.0001,0.0002],
    [0.1511,0.0111,0.0015,0.0002,0.0351,0.1011,0.0101,0.0011,0.0001,0.0015,0.0021,0.4511,0.0511,0.0051,0.0005,0.0411,0.1288,0.0041,0.0004,0.0001,0.0005,0.0008],
    [0.0415,0.0085,0.0008,0.0001,0.0010,0.0012,0.0615,0.0291,0.0035,0.0101,0.0055,0.1508,0.0055,0.0005,0.0001,0.0008,0.0011,0.4849,0.1515,0.0211,0.0105,0.0045],
    [0.0115,0.0015,0.0001,0.0000,0.0001,0.0001,0.0158,0.0901,0.0409,0.0191,0.0085,0.0491,0.0008,0.0000,0.0000,0.0001,0.0001,0.1015,0.4511,0.1555,0.0349,0.0105],
    [0.0022,0.0002,0.0000,0.0000,0.0000,0.0000,0.0035,0.0158,0.1509,0.0851,0.0201,0.0105,0.0001,0.0000,0.0000,0.0000,0.0000,0.0255,0.1015,0.4809,0.0911,0.0115],
    [0.0515,0.0005,0.0001,0.0000,0.0001,0.0002,0.0415,0.0101,0.0015,0.1509,0.0815,0.2515,0.0011,0.0000,0.0000,0.0001,0.0002,0.1815,0.0315,0.0055,0.1499,0.0355],
    [0.1515,0.0015,0.0001,0.0000,0.0002,0.0004,0.0115,0.0015,0.0002,0.0355,0.1011,0.4515,0.0045,0.0001,0.0000,0.0005,0.0008,0.0515,0.0055,0.0005,0.0415,0.1289],
]),
"A1_AUDIT_V3": np.array([
    [0.3105,0.4118,0.0195,0.0195,0.0195,0.0195,0.1266,0.0195,0.0195,0.0195,0.0195,0.0045,0.0045,0.0045,0.0045,0.0045,0.0045,0.0045,0.0045,0.0045,0.0045,0.0045],
    [0.0211,0.1895,0.6526,0.0211,0.0211,0.0211,0.0211,0.0211,0.0211,0.0211,0.0211,0.0021,0.0021,0.0021,0.0021,0.0021,0.0021,0.0021,0.0021,0.0021,0.0021,0.0021],
    [0.0204,0.0204,0.1531,0.7143,0.0204,0.0204,0.0204,0.0204,0.0204,0.0204,0.0204,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020],
    [0.0198,0.0198,0.0198,0.2970,0.5545,0.0198,0.0198,0.0198,0.0198,0.0198,0.0198,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020],
    [0.0217,0.0217,0.0217,0.0217,0.1957,0.6304,0.0217,0.0217,0.0217,0.0217,0.0217,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022],
    [0.8534,0.0251,0.0251,0.0251,0.0251,0.0251,0.0251,0.0251,0.0251,0.0251,0.0251,0.0025,0.0025,0.0025,0.0025,0.0025,0.0025,0.0025,0.0025,0.0025,0.0025,0.0025],
    [0.0211,0.0211,0.0211,0.0211,0.0211,0.0211,0.1895,0.6526,0.0211,0.0211,0.0211,0.0021,0.0021,0.0021,0.0021,0.0021,0.0021,0.0021,0.0021,0.0021,0.0021,0.0021],
    [0.0204,0.0204,0.0204,0.0204,0.0204,0.0204,0.0204,0.1531,0.7143,0.0204,0.0204,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020],
    [0.0198,0.0198,0.0198,0.0198,0.0198,0.0198,0.0198,0.0198,0.2970,0.5545,0.0198,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020,0.0020],
    [0.0217,0.0217,0.0217,0.0217,0.0217,0.0217,0.0217,0.0217,0.0217,0.1957,0.6304,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022,0.0022],
    [0.8534,0.0251,0.0251,0.0251,0.0251,0.0251,0.0251,0.0251,0.0251,0.0251,0.0251,0.0025,0.0025,0.0025,0.0025,0.0025,0.0025,0.0025,0.0025,0.0025,0.0025,0.0025],
    [0.0294,0.0294,0.0294,0.0294,0.0294,0.0294,0.0294,0.0294,0.0294,0.0294,0.0294,0.3235,0.1176,0.0294,0.0294,0.0294,0.0294,0.1471,0.0294,0.0294,0.0294,0.0294],
    [0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0510,0.2041,0.5612,0.0510,0.0510,0.0510,0.0510,0.0510,0.0510,0.0510,0.0510],
    [0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0476,0.0476,0.2381,0.4762,0.0476,0.0476,0.0476,0.0476,0.0476,0.0476,0.0476],
    [0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0444,0.0444,0.0444,0.4000,0.2667,0.0444,0.0444,0.0444,0.0444,0.0444,0.0444],
    [0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0526,0.0526,0.0526,0.0526,0.2105,0.4211,0.0526,0.0526,0.0526,0.0526,0.0526],
    [0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.8125,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625],
    [0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0510,0.0510,0.0510,0.0510,0.0510,0.0510,0.2041,0.5612,0.0510,0.0510,0.0510],
    [0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0476,0.0476,0.0476,0.0476,0.0476,0.0476,0.0476,0.2381,0.4762,0.0476,0.0476],
    [0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0444,0.0444,0.0444,0.0444,0.0444,0.0444,0.0444,0.0444,0.4000,0.2667,0.0444],
    [0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0526,0.0526,0.0526,0.0526,0.0526,0.0526,0.0526,0.0526,0.0526,0.2105,0.4211],
    [0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.8125,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625],
]),
"A1_AUDIT_V4": np.array([
    [0.8105,0.0411,0.0021,0.0020,0.0021,0.0021,0.0423,0.0020,0.0021,0.0021,0.0021,0.0452,0.0041,0.0021,0.0021,0.0021,0.0021,0.0042,0.0021,0.0021,0.0021,0.0021],
    [0.0065,0.7912,0.1503,0.0064,0.0064,0.0064,0.0064,0.0064,0.0064,0.0064,0.0064,0.0064,0.0064,0.0064,0.0064,0.0064,0.0064,0.0064,0.0064,0.0064,0.0064,0.0064],
    [0.0043,0.0043,0.8251,0.1302,0.0043,0.0043,0.0043,0.0043,0.0043,0.0043,0.0043,0.0043,0.0043,0.0043,0.0043,0.0043,0.0043,0.0043,0.0043,0.0043,0.0043,0.0043],
    [0.0029,0.0029,0.0029,0.8953,0.0754,0.0029,0.0029,0.0029,0.0029,0.0029,0.0029,0.0029,0.0029,0.0029,0.0029,0.0029,0.0029,0.0029,0.0029,0.0029,0.0029,0.0029],
    [0.0051,0.0051,0.0051,0.0051,0.7538,0.1897,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051],
    [0.8511,0.0068,0.0068,0.0068,0.0068,0.0068,0.0068,0.0068,0.0068,0.0068,0.0068,0.0471,0.0068,0.0068,0.0068,0.0068,0.0068,0.0068,0.0068,0.0068,0.0068,0.0068],
    [0.0062,0.0062,0.0062,0.0062,0.0062,0.0062,0.7951,0.1553,0.0062,0.0062,0.0062,0.0062,0.0062,0.0062,0.0062,0.0062,0.0062,0.0062,0.0062,0.0062,0.0062,0.0062],
    [0.0045,0.0045,0.0045,0.0045,0.0045,0.0045,0.0045,0.8163,0.1351,0.0045,0.0045,0.0045,0.0045,0.0045,0.0045,0.0045,0.0045,0.0045,0.0045,0.0045,0.0045,0.0045],
    [0.0031,0.0031,0.0031,0.0031,0.0031,0.0031,0.0031,0.0031,0.8864,0.0762,0.0031,0.0031,0.0031,0.0031,0.0031,0.0031,0.0031,0.0031,0.0031,0.0031,0.0031,0.0031],
    [0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.7421,0.1884,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053],
    [0.8603,0.0065,0.0065,0.0065,0.0065,0.0065,0.0065,0.0065,0.0065,0.0065,0.0065,0.0452,0.0065,0.0065,0.0065,0.0065,0.0065,0.0065,0.0065,0.0065,0.0065,0.0065],
    [0.0513,0.0031,0.0031,0.0031,0.0031,0.0031,0.0524,0.0031,0.0031,0.0031,0.0031,0.7891,0.0211,0.0031,0.0031,0.0031,0.0031,0.0223,0.0031,0.0031,0.0031,0.0031],
    [0.0098,0.0098,0.0098,0.0098,0.0098,0.0098,0.0098,0.0098,0.0098,0.0098,0.0098,0.0098,0.7241,0.1265,0.0098,0.0098,0.0098,0.0098,0.0098,0.0098,0.0098,0.0098],
    [0.0071,0.0071,0.0071,0.0071,0.0071,0.0071,0.0071,0.0071,0.0071,0.0071,0.0071,0.0071,0.0071,0.7813,0.1054,0.0071,0.0071,0.0071,0.0071,0.0071,0.0071,0.0071],
    [0.0049,0.0049,0.0049,0.0049,0.0049,0.0049,0.0049,0.0049,0.0049,0.0049,0.0049,0.0049,0.0049,0.0049,0.8421,0.0853,0.0049,0.0049,0.0049,0.0049,0.0049,0.0049],
    [0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0526,0.0526,0.0526,0.0526,0.2105,0.4211,0.0526,0.0526,0.0526,0.0526,0.0526],
    [0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.8125,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625],
    [0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0051,0.0510,0.0510,0.0510,0.0510,0.0510,0.0510,0.2041,0.5612,0.0510,0.0510,0.0510],
    [0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0048,0.0476,0.0476,0.0476,0.0476,0.0476,0.0476,0.0476,0.2381,0.4762,0.0476,0.0476],
    [0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0044,0.0444,0.0444,0.0444,0.0444,0.0444,0.0444,0.0444,0.0444,0.4000,0.2667,0.0444],
    [0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0053,0.0526,0.0526,0.0526,0.0526,0.0526,0.0526,0.0526,0.0526,0.0526,0.2105,0.4211],
    [0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.0063,0.8125,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625,0.0625],
]),
"A1_AUDIT_V5": np.array([
    [0.6512,0.1205,0.0112,0.0015,0.0011,0.0115,0.1211,0.0109,0.0013,0.0010,0.0112,0.0185,0.0055,0.0010,0.0005,0.0010,0.0025,0.0050,0.0010,0.0005,0.0010,0.0025],
    [0.0411,0.4518,0.3501,0.0205,0.0015,0.0011,0.0115,0.0010,0.0005,0.0005,0.0005,0.0555,0.0515,0.0105,0.0010,0.0005,0.0005,0.0005,0.0005,0.0005,0.0005,0.0005],
    [0.0105,0.0511,0.4813,0.3512,0.0115,0.0051,0.0015,0.0005,0.0005,0.0005,0.0005,0.0105,0.0115,0.0513,0.0105,0.0010,0.0005,0.0005,0.0005,0.0005,0.0005,0.0005],
    [0.0015,0.0010,0.0415,0.7511,0.1513,0.0115,0.0005,0.0005,0.0005,0.0005,0.0005,0.0055,0.0010,0.0115,0.0155,0.0055,0.0010,0.0005,0.0005,0.0005,0.0005,0.0005],
    [0.0112,0.0015,0.0010,0.0511,0.4613,0.3811,0.0010,0.0005,0.0005,0.0005,0.0005,0.0105,0.0010,0.0055,0.0115,0.0513,0.0055,0.0005,0.0005,0.0005,0.0005,0.0005],
    [0.4513,0.0115,0.0010,0.0005,0.0411,0.3512,0.0115,0.0010,0.0005,0.0005,0.0005,0.1015,0.0055,0.0010,0.0005,0.0115,0.0055,0.0055,0.0010,0.0005,0.0005,0.0005],
    [0.0411,0.0115,0.0010,0.0005,0.0005,0.0005,0.4518,0.3501,0.0205,0.0015,0.0011,0.0555,0.0005,0.0005,0.0005,0.0005,0.0005,0.0515,0.0105,0.0010,0.0005,0.0005],
    [0.0105,0.0015,0.0005,0.0005,0.0005,0.0005,0.0511,0.4813,0.3512,0.0115,0.0051,0.0105,0.0005,0.0005,0.0005,0.0005,0.0005,0.0115,0.0513,0.0105,0.0010,0.0005],
    [0.0015,0.0005,0.0005,0.0005,0.0005,0.0005,0.0010,0.0415,0.7511,0.1513,0.0115,0.0055,0.0005,0.0005,0.0005,0.0005,0.0005,0.0010,0.0115,0.0155,0.0055,0.0010],
    [0.0112,0.0010,0.0005,0.0005,0.0005,0.0005,0.0015,0.0010,0.0511,0.4613,0.3811,0.0105,0.0005,0.0005,0.0005,0.0005,0.0005,0.0010,0.0055,0.0115,0.0513,0.0055],
    [0.4513,0.0115,0.0005,0.0005,0.0005,0.0005,0.0115,0.0010,0.0005,0.0411,0.3512,0.1015,0.0005,0.0005,0.0005,0.0005,0.0005,0.0055,0.0010,0.0115,0.0055,0.0055],
    [0.0215,0.0045,0.0010,0.0005,0.0010,0.0025,0.0048,0.0010,0.0005,0.0010,0.0025,0.7513,0.0811,0.0055,0.0015,0.0015,0.0055,0.0805,0.0051,0.0013,0.0015,0.0052],
    [0.0105,0.0415,0.0105,0.0010,0.0005,0.0005,0.0005,0.0005,0.0005,0.0005,0.0005,0.0511,0.5513,0.3011,0.0105,0.0015,0.0010,0.0055,0.0010,0.0005,0.0005,0.0005],
    [0.0015,0.0105,0.0413,0.0105,0.0010,0.0005,0.0005,0.0005,0.0005,0.0005,0.0005,0.0115,0.0511,0.5813,0.2511,0.0155,0.0055,0.0010,0.0005,0.0005,0.0005,0.0005],
    [0.0005,0.0010,0.0105,0.0155,0.0055,0.0010,0.0005,0.0005,0.0005,0.0005,0.0005,0.0055,0.0105,0.1511,0.6518,0.1215,0.0105,0.0005,0.0005,0.0005,0.0005,0.0005],
    [0.0010,0.0010,0.0055,0.0115,0.0513,0.0055,0.0005,0.0005,0.0005,0.0005,0.0005,0.0112,0.0015,0.0511,0.1215,0.5613,0.1511,0.0010,0.0005,0.0005,0.0005,0.0005],
    [0.0055,0.0055,0.0010,0.0005,0.0115,0.0055,0.0005,0.0005,0.0005,0.0005,0.0005,0.4011,0.0115,0.0105,0.0105,0.1511,0.3513,0.0115,0.0010,0.0005,0.0005,0.0005],
    [0.0105,0.0005,0.0005,0.0005,0.0005,0.0005,0.0415,0.0105,0.0010,0.0005,0.0005,0.0511,0.0055,0.0010,0.0005,0.0005,0.0005,0.5513,0.3011,0.0105,0.0015,0.0010],
    [0.0015,0.0005,0.0005,0.0005,0.0005,0.0005,0.0105,0.0413,0.0105,0.0010,0.0005,0.0115,0.0010,0.0005,0.0005,0.0005,0.0005,0.0511,0.5813,0.2511,0.0155,0.0055],
    [0.0005,0.0005,0.0005,0.0005,0.0005,0.0005,0.0010,0.0105,0.0155,0.0055,0.0010,0.0055,0.0005,0.0005,0.0005,0.0005,0.0005,0.0105,0.1511,0.6518,0.1215,0.0105],
    [0.0010,0.0005,0.0005,0.0005,0.0005,0.0005,0.0010,0.0055,0.0115,0.0513,0.0055,0.0112,0.0005,0.0005,0.0005,0.0005,0.0005,0.0015,0.0511,0.1215,0.5613,0.1511],
    [0.0055,0.0005,0.0005,0.0005,0.0005,0.0005,0.0055,0.0010,0.0115,0.0055,0.0055,0.4011,0.0005,0.0005,0.0005,0.0005,0.0005,0.0115,0.0105,0.0105,0.1511,0.3513],
]),
"A1_AUDIT_V6": np.array([
    [0.8215,0.0650,0.0025,0.0010,0.0010,0.0015,0.0645,0.0025,0.0010,0.0010,0.0015,0.0150,0.0050,0.0010,0.0005,0.0005,0.0010,0.0050,0.0010,0.0005,0.0005,0.0010],
    [0.1500,0.4550,0.2500,0.0100,0.0050,0.0100,0.0200,0.0010,0.0005,0.0010,0.0015,0.0300,0.0500,0.0100,0.0010,0.0010,0.0010,0.0015,0.0005,0.0000,0.0005,0.0005],
    [0.0100,0.0500,0.5520,0.3500,0.0100,0.0050,0.0010,0.0005,0.0000,0.0005,0.0005,0.0050,0.0050,0.0050,0.0025,0.0010,0.0010,0.0005,0.0000,0.0000,0.0000,0.0000],
    [0.0050,0.0010,0.0500,0.8510,0.0650,0.0100,0.0005,0.0000,0.0000,0.0005,0.0005,0.0025,0.0010,0.0025,0.0050,0.0025,0.0020,0.0005,0.0000,0.0000,0.0000,0.0000],
    [0.1000,0.0100,0.0010,0.0500,0.4030,0.3500,0.0100,0.0010,0.0005,0.0010,0.0015,0.0200,0.0050,0.0005,0.0100,0.0200,0.0150,0.0005,0.0000,0.0000,0.0005,0.0005],
    [0.7515,0.0200,0.0010,0.0050,0.0100,0.1000,0.0500,0.0050,0.0010,0.0050,0.0100,0.0250,0.0025,0.0005,0.0010,0.0025,0.0050,0.0025,0.0010,0.0005,0.0005,0.0010],
    [0.1500,0.0200,0.0010,0.0005,0.0010,0.0015,0.4550,0.2500,0.0100,0.0050,0.0100,0.0300,0.0015,0.0005,0.0000,0.0005,0.0005,0.0500,0.0100,0.0010,0.0010,0.0010],
    [0.0100,0.0010,0.0005,0.0000,0.0005,0.0005,0.0500,0.5520,0.3500,0.0100,0.0050,0.0050,0.0005,0.0000,0.0000,0.0000,0.0000,0.0050,0.0050,0.0025,0.0010,0.0010],
    [0.0050,0.0005,0.0000,0.0000,0.0005,0.0005,0.0010,0.0500,0.8510,0.0650,0.0100,0.0025,0.0005,0.0000,0.0000,0.0000,0.0000,0.0010,0.0025,0.0050,0.0025,0.0020],
    [0.1000,0.0100,0.0005,0.0000,0.0005,0.0005,0.0100,0.0010,0.0500,0.4030,0.3500,0.0200,0.0005,0.0000,0.0000,0.0005,0.0005,0.0050,0.0005,0.0100,0.0200,0.0150],
    [0.7515,0.0500,0.0050,0.0010,0.0050,0.0100,0.0200,0.0010,0.0050,0.0100,0.1000,0.0250,0.0025,0.0010,0.0005,0.0005,0.0010,0.0025,0.0010,0.0010,0.0025,0.0050],
    [0.0150,0.0050,0.0010,0.0005,0.0005,0.0010,0.0050,0.0010,0.0005,0.0005,0.0010,0.7515,0.0850,0.0050,0.0010,0.0015,0.0020,0.0850,0.0050,0.0010,0.0015,0.0020],
    [0.0300,0.0500,0.0100,0.0010,0.0010,0.0010,0.0015,0.0005,0.0000,0.0005,0.0005,0.1000,0.3550,0.4000,0.0100,0.0050,0.0100,0.0200,0.0010,0.0005,0.0010,0.0015],
    [0.0050,0.0050,0.0050,0.0025,0.0010,0.0010,0.0005,0.0000,0.0000,0.0000,0.0000,0.0100,0.0200,0.4020,0.5000,0.0200,0.0100,0.0010,0.0005,0.0000,0.0005,0.0005],
    [0.0025,0.0010,0.0025,0.0050,0.0025,0.0020,0.0005,0.0000,0.0000,0.0000,0.0000,0.0050,0.0050,0.0200,0.8010,0.1250,0.0200,0.0010,0.0005,0.0000,0.0005,0.0005],
    [0.0200,0.0050,0.0005,0.0100,0.0200,0.0150,0.0005,0.0000,0.0000,0.0005,0.0005,0.0500,0.0100,0.0010,0.1000,0.3020,0.4000,0.0100,0.0010,0.0005,0.0010,0.0015],
    [0.0250,0.0025,0.0005,0.0010,0.0025,0.0050,0.0025,0.0010,0.0005,0.0005,0.0010,0.6515,0.0500,0.0010,0.0050,0.0100,0.1500,0.0500,0.0050,0.0010,0.0050,0.0100],
    [0.0300,0.0015,0.0005,0.0000,0.0005,0.0005,0.0500,0.0100,0.0010,0.0010,0.0010,0.1000,0.0200,0.0010,0.0005,0.0010,0.0015,0.3550,0.4000,0.0100,0.0050,0.0100],
    [0.0050,0.0005,0.0000,0.0000,0.0000,0.0000,0.0050,0.0050,0.0025,0.0010,0.0010,0.0100,0.0010,0.0005,0.0000,0.0005,0.0005,0.0200,0.4020,0.5000,0.0200,0.0100],
    [0.0025,0.0005,0.0000,0.0000,0.0000,0.0000,0.0010,0.0025,0.0050,0.0025,0.0020,0.0050,0.0010,0.0005,0.0000,0.0010,0.0010,0.0050,0.0200,0.8010,0.1250,0.0200],
    [0.0200,0.0005,0.0000,0.0000,0.0005,0.0005,0.0050,0.0005,0.0100,0.0200,0.0150,0.0500,0.0005,0.0000,0.0000,0.0005,0.0005,0.0100,0.0010,0.1000,0.3020,0.4000],
    [0.0250,0.0025,0.0010,0.0005,0.0005,0.0010,0.0025,0.0010,0.0010,0.0025,0.0050,0.6515,0.0500,0.0010,0.0005,0.0010,0.0050,0.0500,0.0050,0.0010,0.0100,0.1500],
])
}

In [ ]:
import os, json, math, hashlib, warnings, gc, traceback, importlib.util, random, sys, time
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

FORCE_CPU = True
MIX_ENABLE = True

if FORCE_CPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("OMP_NUM_THREADS","10")
os.environ.setdefault("OPENBLAS_NUM_THREADS","10")
os.environ.setdefault("MKL_NUM_THREADS","10")
os.environ.setdefault("NUMEXPR_NUM_THREADS","10")

SEED = 20250831
np.random.seed(SEED); random.seed(SEED)

try:
    import torch
    if FORCE_CPU:
        device = "cpu"
    else:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(SEED)
    if device == "cuda":
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    torch = None
    device = "cpu"

STAGE1 = Path("STAGE1_OUT")
STAGE2 = Path("STAGE2_OUT")
STAGE3 = Path("STAGE3_OUT"); STAGE3.mkdir(parents=True, exist_ok=True)
EPS = 1e-12
TRAIN_MIN_OBS = 50
BETA_BAYES = 1.25
TARGET_SHARPE = 1.05
BASE_EPOCHS = 60
LR0 = 3e-3
THR_LIST = [0.15, 0.12, 0.10, 0.08, 0.05, 0.0]   # on finit par 0.0 = force d’entrées
BE_LIST  = [0.010, 0.010, 0.008, 0.006, 0.005, 0.003]
SL_PCT = 0.05
KL_W, OFFCYCLE_W, L2_W, BLK_W = 5.0, 10.0, 2.0, 3.0

STATES = [
    "GAS_R","GAS_PTU","GAS_SU","GAS_U","GAS_PEU","GAS_EU","GAS_PTD","GAS_SD","GAS_D","GAS_PED","GAS_ED",
    "FLUIDE_R","FLUIDE_PTU","FLUIDE_SU","FLUIDE_U","FLUIDE_PEU","FLUIDE_EU","FLUIDE_PTD","FLUIDE_SD","FLUIDE_D","FLUIDE_PED","FLUIDE_ED"
]
S2I = {s:i for i,s in enumerate(STATES)}
UP_TAG, DN_TAG = {"PTU","SU","U"}, {"PTD","SD","D"}
UP_IDX = np.array([S2I[s] for s in STATES if s.split("_",1)[1] in UP_TAG], dtype=np.int64)
DN_IDX = np.array([S2I[s] for s in STATES if s.split("_",1)[1] in DN_TAG], dtype=np.int64)

def allowed_edges():
    E=set(); base=["R","PTU","SU","U","PEU","EU","R","PTD","SD","D","PED","ED","R"]
    for reg in ("GAS","FLUIDE"):
        seq=[f"{reg}_{t}" for t in base]
        for a,b in zip(seq[:-1],seq[1:]): E.add((a,b))
        for t in base[:-1]: E.add((f"{reg}_{t}",f"{reg}_{t}"))
    return E
ALLOWED = {(S2I[a],S2I[b]) for (a,b) in allowed_edges()}

def _sanitize(P: np.ndarray)->np.ndarray:
    A = np.array(P, dtype=np.float64); A[A<EPS]=EPS; A/=A.sum(axis=1, keepdims=True); return A
def mhash(P: np.ndarray)->str: return hashlib.sha1(np.ascontiguousarray(P,dtype=np.float64)).hexdigest()[:12]
def _sym_tag(x:str)->str: return x.upper().replace('.','_').replace('-','_')

def load_ser_matrices() -> Dict[str,np.ndarray]:
    if "SER_MATRICES" in globals():
        mats = globals()["SER_MATRICES"]
        if isinstance(mats, dict) and len(mats)==6:
            return {k:_sanitize(np.array(v, dtype=np.float64)) for k,v in mats.items()}
    p = STAGE2 / "SER_MATRICES.json"
    if p.exists():
        with open(p,"r",encoding="utf-8") as f: mats=json.load(f)
        return {k:_sanitize(np.array(v, dtype=np.float64)) for k,v in mats.items()}
    if Path("SER_MATRICES.py").exists():
        spec = importlib.util.spec_from_file_location("SER_MATRICES","SER_MATRICES.py")
        mod  = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)  # type: ignore
        mats = getattr(mod,"SER_MATRICES",None)
        if isinstance(mats, dict) and len(mats)==6:
            return {k:_sanitize(np.array(v, dtype=np.float64)) for k,v in mats.items()}
    raise RuntimeError("SER_MATRICES introuvable (Cellule 2).")
SER_MATRICES = load_ser_matrices()

def load_manifest_paths(mfile: Path)->Tuple[str,Dict[str,str]]:
    with open(mfile,"r",encoding="utf-8") as f: man=json.load(f)
    sym = man.get("symbol") or mfile.name.split("__")[0]
    return sym, man.get("paths",{})

def load_ohlc_daily(paths: Dict[str,str]) -> pd.DataFrame:
    p = paths.get("OHLC_1Y") or paths.get("OHLC_10Y", None)
    if not p: raise RuntimeError("OHLC 1Y manquant.")
    df = pd.read_parquet(p) if str(p).endswith(".parquet") else pd.read_csv(p, index_col=0, parse_dates=True)
    if "date" in df.columns: df["date"]=pd.to_datetime(df["date"]); df=df.set_index("date")
    return df.rename(columns={c:c.lower() for c in df.columns})

def load_daily1(paths: Dict[str,str]) -> pd.DataFrame:
    p = paths.get("Daily_1Y", None)
    if not p: raise RuntimeError("Daily_1Y manquant (features).")
    df = pd.read_parquet(p) if str(p).endswith(".parquet") else pd.read_csv(p, index_col=0, parse_dates=True)
    return df

def palo_alto_from_daily1(d1: pd.DataFrame) -> pd.Series:
    df = d1.copy()
    mu  = pd.to_numeric(df.get("mu"), errors="coerce")
    sig = pd.to_numeric(df.get("sigma"), errors="coerce").replace(0,np.nan)
    ts  = (mu.abs()/sig).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    ent = pd.to_numeric(
        df.filter(regex="ENT.*entropy_total").iloc[:,0] if len(df.filter(regex="ENT.*entropy_total").columns)>0
        else pd.Series(index=df.index), errors="coerce"
    )
    pnF = pd.to_numeric(df.get("premice_to_FLUIDE_score"), errors="coerce")
    pnG = pd.to_numeric(df.get("premice_to_GAS_score"), errors="coerce")
    reg = df.get("regime")
    if reg is None or reg.dropna().empty:
        e = (ent - ent.expanding(10).median())/((ent.expanding(10).std(ddof=1)+1e-6))
        reg = pd.Series(np.where(e>0.0,"FLUIDE","GAS"), index=df.index)

    ts_q = ts.expanding(120).quantile
    up1  = ts_q(0.60).fillna(ts.median())
    up2  = ts_q(0.80).fillna(ts.quantile(0.8))
    ent_n = (ent - ent.expanding(60).min())/((ent.expanding(60).max()-ent.expanding(60).min()).replace(0,np.nan))
    clarity = (1.0 - ent_n).clip(0,1).fillna(0.5)
    tags = pd.Series(index=df.index, dtype=object)
    tags[(ts<0.35) & (clarity>0.4)] = "R"
    tags[tags.isna() & (pnG>0.6) & (mu.between(mu.expanding(20).quantile(0.3),0.0))] = "PTD"
    tags[tags.isna() & (pnF>0.6) & (mu.between(0.0, mu.expanding(20).quantile(0.7)))] = "PTU"
    tags[tags.isna() & (mu<0) & (ts>up1) & (clarity>0.45)] = "SD"
    tags[tags.isna() & (mu>0) & (ts>up1) & (clarity>0.45)] = "SU"
    tags[tags.isna() & (ts>up2)] = np.where(mu>0,"U","D")
    dts = ts.diff().fillna(0.0)
    tags[tags.isna() & (mu>0) & (dts<0) & (ent_n>0.5)] = "PEU"
    tags[tags.isna() & (mu<0) & (dts>0) & (ent_n>0.5)] = "PED"
    tags[tags.isna() & (mu>0) & (ts<0.1)] = "EU"
    tags[tags.isna() & (mu<0) & (ts<0.1)] = "ED"
    tags = tags.fillna("R")
    states = pd.Series([f"{reg.iloc[i]}_{tags.iloc[i]}" for i in range(len(tags))], index=df.index)
    return states[states.isin(STATES)]

def eig2_numpy(P: np.ndarray)->float:
    try:
        w=np.linalg.eigvals(P); w=np.sort(np.abs(w))[::-1]
        return float(w[1]) if len(w)>1 else 0.0
    except Exception:
        return np.nan

if torch is not None:
    class ParamP(torch.nn.Module):
        def __init__(self, P0: np.ndarray, kl_w=KL_W, off_w=OFFCYCLE_W, l2_w=L2_W, blk_w=BLK_W):
            super().__init__()
            self.logits = torch.nn.Parameter(torch.log(torch.tensor(P0+1e-12, dtype=torch.float32)))
            self.Pref   = torch.tensor(P0, dtype=torch.float32, requires_grad=False)
            self.kl_w, self.off_w, self.l2_w, self.blk_w = kl_w, off_w, l2_w, blk_w
            self.allowed = torch.zeros((22,22), dtype=torch.bool)
            for (i,j) in ALLOWED: self.allowed[i,j]=True
            gas = [i for i,s in enumerate(STATES) if s.startswith("GAS_")]
            flu = [i for i,s in enumerate(STATES) if s.startswith("FLUIDE_")]
            self.gidx=torch.tensor(gas); self.fidx=torch.tensor(flu)

        def get_P(self)->torch.Tensor: return torch.softmax(self.logits, dim=1)

        def penalties(self)->torch.Tensor:
            P = self.get_P()
            kl = torch.sum(torch.sum(P * (torch.log((P+1e-12)/(self.Pref+1e-12))), dim=1))
            off = torch.sum(P[~self.allowed])
            gg = (P[self.gidx][:, self.gidx].sum()); gf = (P[self.gidx][:, self.fidx].sum())
            fg = (P[self.fidx][:, self.gidx].sum()); ff = (P[self.fidx][:, self.fidx].sum())
            gg0 = (self.Pref[self.gidx][:, self.gidx].sum()); gf0 = (self.Pref[self.gidx][:, self.fidx].sum())
            fg0 = (self.Pref[self.fidx][:, self.gidx].sum()); ff0 = (self.Pref[self.fidx][:, self.fidx].sum())
            blk = ( (gg-gg0)**2 + (gf-gf0)**2 + (fg-fg0)**2 + (ff-ff0)**2 )
            with torch.no_grad():
                lam = abs(eig2_numpy(P.detach().cpu().numpy()) - eig2_numpy(self.Pref.detach().cpu().numpy()))
            l2 = torch.tensor(lam, dtype=torch.float32, device=P.device)
            return KL_W*kl + OFFCYCLE_W*off + BLK_W*blk + L2_W*l2

    class MLPBias(torch.nn.Module):
        def __init__(self, in_dim: int):
            super().__init__()
            self.net = torch.nn.Sequential(
                torch.nn.Linear(in_dim, 64), torch.nn.GELU(),
                torch.nn.Linear(64, 32), torch.nn.GELU(),
                torch.nn.Linear(32, 22)
            )
        def forward(self, x)->torch.Tensor: return self.net(x)

def sharpe_loss(pnl: "torch.Tensor")->"torch.Tensor":
    mu = pnl.mean(); sd = pnl.std(unbiased=False) + 1e-8
    return -(mu / sd)

def run_trading_discrete(dates, p_long, p_short, p_entryL, p_entryS, ohlc, states_labels,
                         thr_entry, sl_pct, be_trigger):
    pnl_day = pd.Series(0.0, index=pd.Index(dates, name="date"))
    trades = []
    pos, entry, stop = 0, np.nan, np.nan
    for i in range(len(dates)-1):
        d = dates[i]; nxt = dates[i+1]
        c0 = float(ohlc.loc[d,"close"]); c1=float(ohlc.loc[nxt,"close"])
        h1 = float(ohlc.loc[nxt,"high"]);  l1=float(ohlc.loc[nxt,"low"])
        lab_next = states_labels[i+1].split("_",1)[1]

        if pos==0:
            if (p_long[i]>=thr_entry) and (p_entryL[i]>=thr_entry):
                pos=+1; entry=c0; stop=entry*(1.0-sl_pct)
                trades.append([d,"OPEN_LONG",entry,0.0,lab_next])
            elif (p_short[i]>=thr_entry) and (p_entryS[i]>=thr_entry):
                pos=-1; entry=c0; stop=entry*(1.0+sl_pct)
                trades.append([d,"OPEN_SHORT",entry,0.0,lab_next])
            continue

        if pos>0:
            if (h1-entry)/max(entry,EPS)>=be_trigger: stop=max(stop,entry)
            if l1<=stop:
                pnl = stop - c0; pnl_day.loc[nxt]=pnl
                trades.append([nxt,"STOP_LONG",stop,pnl,lab_next])
                pos=0; entry=np.nan; stop=np.nan; continue
            if lab_next in DN_TAG:
                pnl = c1 - c0; pnl_day.loc[nxt]=pnl
                trades.append([nxt,"EXIT_LONG_REGIME",c1,pnl,lab_next])
                pos=0; entry=np.nan; stop=np.nan; continue
            pnl = c1 - c0; pnl_day.loc[nxt]=pnl
        else:
            if (entry-l1)/max(entry,EPS)>=be_trigger: stop=min(stop,entry)
            if h1>=stop:
                pnl = c0 - stop; pnl_day.loc[nxt]=pnl
                trades.append([nxt,"STOP_SHORT",stop,pnl,lab_next])
                pos=0; entry=np.nan; stop=np.nan; continue
            if lab_next in UP_TAG:
                pnl = c0 - c1; pnl_day.loc[nxt]=pnl
                trades.append([nxt,"EXIT_SHORT_REGIME",c1,pnl,lab_next])
                pos=0; entry=np.nan; stop=np.nan; continue
            pnl = c0 - c1; pnl_day.loc[nxt]=pnl
    tdf = pd.DataFrame(trades, columns=["date","event","price","pnl","phase_next"])
    if not tdf.empty: tdf["date"]=pd.to_datetime(tdf["date"])
    return pnl_day, tdf

def sharpe_points(s: pd.Series)->float:
    return float((s.mean()/(s.std(ddof=0)+1e-8))*np.sqrt(252.0)) if not s.empty else 0.0
def maxdd_points(c: pd.Series)->float:
    rm=c.cummax(); dd=c-rm; return float(dd.min()) if not c.empty else 0.0

def _train_inner(X_t, y_t, st_t, dates, ohlc, st, Pref, label, verbose=False):
    in_dim = X_t.shape[1]
    model  = MLPBias(in_dim).to(device)
    pparam = ParamP(Pref).to(device)
    opt    = torch.optim.Adam(list(model.parameters())+list(pparam.parameters()), lr=LR0)

    N = X_t.shape[0]
    train_len = max(int(0.6*N), min(200,int(0.7*N)))

    best_sh=-9e9; best_pack=None
    for kick,(thr,be) in enumerate(zip(THR_LIST, BE_LIST)):
        if verbose: print(f"      [EP] thr={thr:.3f} be={be:.3f} …", flush=True)
        for ep in range(BASE_EPOCHS):
            i0 = max(0, N-train_len-1); i1 = N-2
            xw = X_t[i0:i1]; yw = y_t[i0:i1]; stw = st_t[i0:i1]
            logits_bias = model(xw)
            P = pparam.get_P()
            rowP = P[stw]
            logits = torch.log(rowP+1e-12) + logits_bias
            pnext = torch.softmax(logits, dim=1)
            plong  = torch.sum(pnext[:, UP_IDX], dim=1)
            pshort = torch.sum(pnext[:, DN_IDX], dim=1)
            expo = torch.tanh(1.25*(plong-pshort))
            pnl  = expo * yw
            loss = sharpe_loss(pnl) + pparam.penalties()
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(list(model.parameters())+list(pparam.parameters()), max_norm=3.0)
            opt.step()
            if verbose and (ep+1) % 15 == 0:
                with torch.no_grad():
                    sh_tmp = float((pnl.mean()/(pnl.std(unbiased=False)+1e-8))*np.sqrt(252.0))
                print(f"         - epoch {ep+1}/{BASE_EPOCHS} | Sharpe(train)~{sh_tmp:.2f}", flush=True)

        with torch.no_grad():
            lb = model(X_t[:-1])
            P  = pparam.get_P()
            rowP = P[st_t[:-1]]
            logits = torch.log(rowP+1e-12) + lb
            pnext = torch.softmax(logits, dim=1).detach().cpu().numpy()
            plong = pnext[:, UP_IDX].sum(1)
            pshort= pnext[:, DN_IDX].sum(1)
            peL   = pnext[:, S2I["GAS_PTU"]] + pnext[:, S2I["FLUIDE_PTU"]]
            peS   = pnext[:, S2I["GAS_PTD"]] + pnext[:, S2I["FLUIDE_PTD"]]
            labs  = st.values.tolist()

        pnl_day, trades = run_trading_discrete(dates[:-1], plong, pshort, peL, peS, ohlc, labs, thr, SL_PCT, be)
        closed = trades[trades["event"].str.startswith(("STOP_","EXIT_"))]
        if closed.empty and kick < len(THR_LIST)-1:
            continue

        sh = sharpe_points(pnl_day)
        if sh > best_sh:
            best_sh = sh
            best_pack = (pnl_day.cumsum(), pnl_day, trades, P.detach().cpu().numpy())

        if sh >= TARGET_SHARPE:
            break

    if best_pack is None:
        return 0.0, 0.0, 0.0, 0, Pref
    curve, pnl, trd, Padapt = best_pack
    mdd = maxdd_points(curve)
    wr  = float((trd["pnl"]>0).mean()) if not trd.empty else 0.0
    return best_sh, mdd, wr, int((trd["event"].str.startswith(("STOP_","EXIT_"))).sum()), Padapt

def train_for_matrix(sym:str, ohlc:pd.DataFrame, d1:pd.DataFrame, states:pd.Series,
                     Pref:np.ndarray, label:str, verbose=True) -> Tuple[float,float,float,int,np.ndarray]:
    idx = d1.index.intersection(ohlc.index).intersection(states.index)
    o = ohlc.loc[idx]; d = d1.loc[idx]; st = states.loc[idx]

    mu  = pd.to_numeric(d.get("mu"), errors="coerce").fillna(0.0)
    sg0 = pd.to_numeric(d.get("sigma"), errors="coerce").replace(0,np.nan)
    sg  = sg0.fillna(method="ffill").fillna(sg0.median() if np.isfinite(sg0.median()) else 0.0)
    ts  = (mu.abs()/sg).replace([np.inf,-np.inf], np.nan).fillna(0.0)
    H   = pd.to_numeric(d.filter(regex="Hurst").iloc[:,0] if len(d.filter(regex="Hurst").columns)>0 else pd.Series(index=idx), errors="coerce").fillna(0.5)
    L   = pd.to_numeric(d.filter(regex="Lyap").iloc[:,0] if len(d.filter(regex="Lyap").columns)>0 else pd.Series(index=idx), errors="coerce").fillna(0.0)
    ent = pd.to_numeric(d.filter(regex="ENT.*entropy_total").iloc[:,0] if len(d.filter(regex="ENT.*entropy_total").columns)>0 else pd.Series(index=idx), errors="coerce").fillna(0.0)
    pnF = pd.to_numeric(d.get("premice_to_FLUIDE_score"), errors="coerce").fillna(0.5)
    pnG = pd.to_numeric(d.get("premice_to_GAS_score"), errors="coerce").fillna(0.5)

    sti = st.map(S2I).astype(int)
    oh = np.zeros((len(sti),22),dtype=np.float32); oh[np.arange(len(sti)), sti.values]=1.0

    X = np.stack([mu.values, sg.values, ts.values, H.values, L.values, ent.values, pnF.values, pnG.values], axis=1).astype(np.float32)
    X = np.concatenate([X, oh], axis=1)
    y = (o["close"].shift(-1)-o["close"]).astype(np.float32).fillna(0.0).values
    dates = list(idx)

    if len(idx)<TRAIN_MIN_OBS or torch is None:
        return 0.0, 0.0, 0.0, 0, Pref

    X_t = torch.tensor(X, dtype=torch.float32, device=device)
    y_t = torch.tensor(y, dtype=torch.float32, device=device)
    st_t= torch.tensor(sti.values, dtype=torch.long, device=device)

    try:
        return _train_inner(X_t,y_t,st_t,dates,ohlc.loc[idx],st,Pref,label,verbose=verbose)
    except Exception as e:
        if (device == "cuda"):
            print(f"      [FALLBACK→CPU] {type(e).__name__}: {e}", flush=True)
            cpu = "cpu"
            X_t = torch.tensor(X, dtype=torch.float32, device=cpu)
            y_t = torch.tensor(y, dtype=torch.float32, device=cpu)
            st_t= torch.tensor(sti.values, dtype=torch.long, device=cpu)
            return _train_inner(X_t,y_t,st_t,dates,ohlc.loc[idx],st,Pref,label,verbose=verbose)
        else:
            raise

def run_symbol(mfile: Path):
    sym, paths = load_manifest_paths(mfile)
    try:
        print(f"\n[LOAD] {sym} | device={device} | torch={'ON' if torch is not None else 'OFF'}", flush=True)
        ohlc = load_ohlc_daily(paths)
        d1   = load_daily1(paths)
        st   = palo_alto_from_daily1(d1)
        idx  = d1.index.intersection(ohlc.index).intersection(st.index)
        if len(idx)<TRAIN_MIN_OBS:
            print(f"[SKIP] {sym}: historique utile {len(idx)}<{TRAIN_MIN_OBS}.", flush=True)
            return sym, None
        ohlc = ohlc.loc[idx]; d1=d1.loc[idx]; st=st.loc[idx]

        results=[]; post=[]
        for name,P0 in SER_MATRICES.items():
            print(f"[WF]  {sym} | test {name} ({mhash(P0)})", flush=True)
            sh,mdd,wr,nt,Pad = train_for_matrix(sym, ohlc, d1, st, P0, name, verbose=True)
            print(f"      -> Sharpe={sh:.2f} | MaxDD={mdd:.2f} | WR={wr*100:.1f}% | Trades={nt}", flush=True)
            results.append((name,sh,mdd,wr,nt,Pad)); post.append(np.exp(BETA_BAYES*sh))
        post = np.array(post); post/=post.sum() if post.sum()>0 else 1.0

        i_best = int(np.argmax([x[1] for x in results]))
        best_name,best_sh,best_mdd,best_wr,best_nt,best_P = results[i_best]

        sh_mix = float("nan")
        if MIX_ENABLE:
            Pmix = np.zeros_like(best_P);
            for k,(_,sh,_,_,_,Pad) in enumerate(results): Pmix += post[k]*Pad
            st_i = st.map(S2I).astype(int).values[:-1]
            row = Pmix[st_i]; pnx = row
            pl = pnx[:, UP_IDX].sum(1); ps=pnx[:, DN_IDX].sum(1)
            peL = pnx[:,S2I["GAS_PTU"]] + pnx[:,S2I["FLUIDE_PTU"]]
            peS = pnx[:,S2I["GAS_PTD"]] + pnx[:,S2I["FLUIDE_PTD"]]
            pnl_day,_ = run_trading_discrete(list(idx)[:-1], pl, ps, peL, peS, ohlc, st.values.tolist(), THR_LIST[0], SL_PCT, BE_LIST[0])
            sh_mix = sharpe_points(pnl_day)

        st_last=st.iloc[-1]; reg,tag=st_last.split("_",1)
        friendly={"R":"Range","PTU":"Prémices U","SU":"Début U fort","U":"Tendance U",
                  "PEU":"Prémices fin U","EU":"Fin U","PTD":"Prémices D","SD":"Début D fort",
                  "D":"Tendance D","PED":"Prémices fin D","ED":"Fin D"}
        print(f"[RESULT] {sym} | Matrice={best_name} ({mhash(SER_MATRICES[best_name])}) | Régime={reg} — Phase={friendly.get(tag,tag)}", flush=True)
        print(f"         Sharpe(points)={best_sh:.2f} | MaxDD(points)={best_mdd:.2f} | Winrate={best_wr*100:.1f}% | Trades={best_nt}", flush=True)
        if MIX_ENABLE: print(f"         Mix(π) Sharpe={sh_mix:.2f} | π={np.round(post,3)}", flush=True)

        with open(STAGE3/f"{_sym_tag(sym)}__summary.json","w",encoding="utf-8") as f:
            json.dump({
                "symbol":sym,
                "best":best_name,
                "Sharpe_points":best_sh,
                "MaxDD_points":best_mdd,
                "Winrate":best_wr,
                "Trades":best_nt,
                "posterior":{nm:float(post[k]) for k,(nm,_,_,_,_,_) in enumerate(results)},
                "mix_sharpe":sh_mix
            }, f, indent=2)
        return sym, (best_name,best_sh,best_mdd,best_wr,best_nt)
    except Exception as e:
        print(f"[ERROR] {sym}: {type(e).__name__}: {e}", flush=True)
        traceback.print_exc()
        return sym, None
    finally:
        gc.collect()

def main():
    print(f"[RUN] NN Bayésien + Adaptation | device={device}", flush=True)
    if not STAGE1.exists():
        raise RuntimeError("STAGE1_OUT introuvable. Exécute la Cellule 1.")
    manifests = sorted(STAGE1.glob("*__manifest.json"))
    print(f"[DISCOVER] manifests trouvés: {len(manifests)}", flush=True)
    if not manifests:
        for p in STAGE1.iterdir(): print("  ", p.name, flush=True)
        raise RuntimeError("Aucun manifest dans STAGE1_OUT. Exécute la Cellule 1.")

    summary=[]
    for mf in manifests:
        sym,res = run_symbol(mf)
        if res: summary.append((sym,)+res)

    if summary:
        df = pd.DataFrame(summary, columns=["symbol","matrix","Sharpe_points","MaxDD_points","Winrate","Trades"])
        df = df.sort_values("Sharpe_points", ascending=False)
        print("\n=== RÉCAPITULATIF ===", flush=True)
        print(df.to_string(index=False), flush=True)
        df.to_csv(STAGE3/"summary.csv", index=False)
    else:
        print("[DONE] Aucun résultat exploitable.", flush=True)

if __name__ == "__main__":
    print("[BOOT] Cellule 3 (SAFE) — démarrage…", flush=True)
    t0=time.time()
    main()
    print(f"[END] Durée {time.time()-t0:.2f}s", flush=True)
